# 06 - Evaluation: all three tiers on all 30 questions

This is where the project's result comes from. Every question goes through
Tier 1, Tier 2 and Tier 3, scored the same way.

The two gaps are what we are measuring:

- **Tier 1 to Tier 2** - what using real data is worth
- **Tier 2 to Tier 3** - what checking is worth

This takes a while on a laptop: 15 questions x 3 tiers, plus Tier 3 retries.


## Setup

All the code from notebooks 04 and 05, so this notebook runs on its own.


In [3]:
import json
import re
import time

import duckdb
import sqlglot
from sqlglot import expressions as exp

DB = "../data/processed/evidenceiq.duckdb"
MODEL = "gemma3:4b"          # Abdelateef used gemma3:12b - use what you have
TOLERANCE = 0.01


# We never let the agent write to the database. Two safety nets: this check,
# and opening DuckDB with read_only=True.
#
# We parse the SQL into a syntax tree instead of searching for banned words.
# Word matching is easy to fool - a column called "updated_at" contains
# "update" - and it misses things a parser catches for free.

FORBIDDEN = {"insert", "update", "delete", "create", "drop", "alter", "copy",
             "command", "merge", "truncate", "attach", "detach", "install",
             "load", "pragma"}


def check_sql(sql):
    """Parse one read-only query; DuckDB also blocks writes and external files."""
    if not isinstance(sql, str) or not sql.strip():
        return "the query is empty"
    try:
        statements = sqlglot.parse(sql, read="duckdb")
        if len(statements) != 1 or statements[0] is None:
            return "only one SELECT query is allowed"
        tree = statements[0]
        if not isinstance(tree, exp.Query):
            return "only SELECT queries are allowed"
        for node in tree.walk():
            if getattr(node, "key", "").lower() in FORBIDDEN:
                return "forbidden operation: %s" % node.key
    except Exception as error:
        return "could not parse the SQL: %s" % error
    return None


def add_to_log(log, tool, code, ok, rows=None, error=None):
    """Every tool call goes in the log. The verifier only trusts what's here."""
    call = {"n": len(log), "tool": tool, "code": code,
            "ok": ok, "rows": rows or [], "error": error}
    log.append(call)
    return call


def numbers_in_log(log):
    """SQL is the root evidence. Calculations must trace back to earlier evidence."""
    import math
    from decimal import Decimal
    found = []
    for call in log or []:
        if not isinstance(call, dict) or not call.get("ok"):
            continue
        for row in call.get("rows") or []:
            if not isinstance(row, dict):
                continue
            if call.get("tool") == "run_python":
                try:
                    inputs = row.get("inputs")
                    result, _ = calculate(row.get("operation"), inputs)
                    # Matching arbitrary model inputs would turn invented values into evidence.
                    if not inputs or not all(any(math.isclose(float(x), v, rel_tol=1e-6, abs_tol=.01)
                                                 for _, v in found) for x in inputs):
                        continue
                    if not math.isclose(float(row["value"]), result, rel_tol=1e-6, abs_tol=.0001):
                        continue
                    values = [row["value"]]
                except (TypeError, ValueError, KeyError, OverflowError):
                    continue
            elif call.get("tool") == "run_sql":
                values = [value for key, value in row.items() if key not in ('customer_id', 'stock_code', 'invoice_no')]
            else:
                continue  # chart metadata is not numerical evidence
            for value in values:
                if isinstance(value, (int, float, Decimal)) and not isinstance(value, bool):
                    if math.isfinite(float(value)):
                        found.append((call.get("n"), float(value)))
    return found


def show_results(log, start=0):
    """Turn the log into text we can paste into the next prompt."""
    if len(log) <= start:
        return "(no queries were run)"
    text = ""
    for call in log[start:]:
        text += "[call %d] %s\n%s\n" % (call["n"], call["tool"], str(call["code"]).strip())
        text += "-> %s\n\n" % (json.dumps(call["rows"][:20]) if call["ok"]
                                else "FAILED: " + str(call["error"]))
    return text


def sql_metadata(sql):
    """Which columns and filters did this query actually use?

    Two of the seven things we have to show the user are the fields and the
    filters. We read them off the query itself rather than asking the model,
    because the model will happily make them up.

    Aliases are skipped: in "SUM(revenue) AS total_revenue" the real field is
    revenue, not total_revenue.
    """
    try:
        tree = sqlglot.parse_one(sql, read="duckdb")
    except Exception:
        return [], []

    aliases = {a.alias for a in tree.find_all(exp.Alias) if a.alias}

    fields = []
    for column in tree.find_all(exp.Column):
        name = column.name
        if name and name not in aliases and name not in fields:
            fields.append(name)

    filters = []
    for where in tree.find_all(exp.Where):
        text = where.this.sql(dialect="duckdb")
        if text not in filters:
            filters.append(text)

    return fields, filters


def json_safe(value):
    """Keep decimal aggregates numeric and dates readable in the evidence log."""
    from decimal import Decimal
    if isinstance(value, Decimal):
        return float(value)
    if hasattr(value, "isoformat"):
        return value.isoformat()
    if hasattr(value, "item"):
        return value.item()
    return value


def query_scope_problem(sql, question):
    """Check known business populations and date scopes, not arbitrary SQL meaning.

    Case/filtered aggregates may restrict dates without an outer WHERE. Their
    conditions count as date restrictions, but a date in an alias or SELECT
    literal does not. These guards supplement execution and evidence checks.
    """
    tree = sqlglot.parse_one(sql, read='duckdb')
    contract = question_contract(question)
    if contract:
        expected = sqlglot.parse_one(contract['sql'], read='duckdb')
        if tree != expected:
            return 'This question has a checked KPI query pattern. Use: ' + contract['sql']
        return None

    country_problem = country_scope_problem(sql, question)
    if country_problem:
        return country_problem

    text = (question or '').lower()
    years = set(re.findall(r'\b(?:19|20)\d{2}\b', text))
    tables = {table.name.lower() for table in tree.find_all(exp.Table)}
    # Month summaries have one row per month; sales has many invoice lines.
    # Joining without the month key repeats the entire fact table, and summing
    # month-level measures after a line-level join multiplies those measures.
    fact_relations, raw_fact_relations = {'sales'}, {'sales'}
    month_relations = {'dim_month'}

    def source_kind(source):
        if isinstance(source, exp.Table):
            name = source.name.lower()
            return name in fact_relations, name in raw_fact_relations, name in month_relations
        if isinstance(source, exp.Subquery):
            return select_kind(source.this)
        return False, False, False

    def select_kind(select):
        if not isinstance(select, exp.Select):
            return False, False, False
        source = select.args.get('from_')
        relations = ([source.this] if source is not None else []) + [join.this for join in select.args.get('joins', [])]
        kinds = [source_kind(relation) for relation in relations]
        fact = any(kind[0] for kind in kinds)
        group = select.args.get('group')
        # Only one row per invoice_month removes the line-level duplication.
        one_row_per_month = bool(group and len(group.expressions) == 1
                                 and isinstance(group.expressions[0], exp.Column)
                                 and group.expressions[0].name.lower() == 'invoice_month')
        return fact, fact and not one_row_per_month, any(kind[2] for kind in kinds)

    for cte in tree.find_all(exp.CTE):
        fact, raw_fact, month = select_kind(cte.this)
        name = cte.alias_or_name.lower()
        if fact:
            fact_relations.add(name)
        if raw_fact:
            raw_fact_relations.add(name)
        if month:
            month_relations.add(name)

    def links_month(predicate, fact_aliases, month_alias):
        if isinstance(predicate, (exp.Where, exp.Paren)):
            return links_month(predicate.this, fact_aliases, month_alias)
        if isinstance(predicate, exp.And):
            return any(links_month(part, fact_aliases, month_alias) for part in (predicate.this, predicate.expression))
        if isinstance(predicate, exp.Or):
            return all(links_month(part, fact_aliases, month_alias) for part in (predicate.this, predicate.expression))
        if isinstance(predicate, exp.EQ):
            left, right = predicate.this, predicate.expression
            if isinstance(left, exp.Column) and isinstance(right, exp.Column) and left.name.lower() == right.name.lower() == 'invoice_month':
                return ((left.table.lower() in fact_aliases and right.table.lower() == month_alias)
                        or (right.table.lower() in fact_aliases and left.table.lower() == month_alias))
        return False

    for select in tree.find_all(exp.Select):
        source = select.args.get('from_')
        relations = ([source.this] if source is not None else []) + [join.this for join in select.args.get('joins', [])]
        classified = [(relation.alias_or_name.lower(), source_kind(relation)) for relation in relations]
        fact_aliases = {alias for alias, kind in classified if kind[0]}
        month_aliases = {alias for alias, kind in classified if kind[2]}
        if not fact_aliases or not month_aliases:
            continue
        clauses = ([select.args['where']] if select.args.get('where') is not None else [])
        clauses += [join.args['on'] for join in select.args.get('joins', []) if join.args.get('on') is not None]
        for month_alias in month_aliases:
            using_month = any((join.this.alias_or_name.lower() == month_alias
                               or (join.this.alias_or_name.lower() in fact_aliases and source is not None and source.this.alias_or_name.lower() == month_alias))
                              and any(identifier.name.lower() == 'invoice_month' for identifier in join.args.get('using') or [])
                              for join in select.args.get('joins', []))
            if not using_month and not any(links_month(clause, fact_aliases, month_alias) for clause in clauses):
                return ('A sales/month join must link the matching month: sales_alias.invoice_month = month_alias.invoice_month. '
                        'ON 1=1 or a date filter only on dim_month repeats sales from unrelated months. '
                        'For overall monthly revenue/per-day comparisons, remove sales and query dim_month directly.')
        if any(kind[1] for _, kind in classified):
            for aggregate in select.find_all(exp.Sum, exp.Avg):
                if aggregate.find_ancestor(exp.Select) is not select:
                    continue
                if any(column.name.lower() in {'trading_days', 'net_revenue', 'gross_revenue'}
                       and (not column.table or column.table.lower() in month_aliases)
                       for column in aggregate.find_all(exp.Column)):
                    return ('Do not SUM or AVG dim_month measures after joining invoice lines: this repeats trading_days/monthly revenue once per line. '
                            'For overall monthly analysis use dim_month alone. For filtered sales, group sales by invoice_month first, '
                            'then join that one-row-per-month result to dim_month on invoice_month.')

    wheres = list(tree.find_all(exp.Where))
    filters = ' '.join(where.sql(dialect='duckdb') for where in wheres)
    normalized = re.sub(r'\b\w+\.', '', filters.lower())
    normalized = re.sub(r'\s+', ' ', normalized)
    temporal_columns = {'invoice_date', 'invoice_month'}

    def predicate_years(predicate):
        # A year in one OR branch does not restrict the other branch. Similarly,
        # NOT(year = ...) selects other dates, not the named reporting year.
        if isinstance(predicate, (exp.Where, exp.Paren)):
            return predicate_years(predicate.this)
        if isinstance(predicate, exp.And):
            return predicate_years(predicate.this) | predicate_years(predicate.expression)
        if isinstance(predicate, exp.Or):
            left, right = predicate_years(predicate.this), predicate_years(predicate.expression)
            return left | right if left and right else set()
        if isinstance(predicate, exp.Not):
            return set()
        if not any(col.name.lower() in temporal_columns for col in predicate.find_all(exp.Column)):
            return set()
        return {year for literal in predicate.find_all(exp.Literal)
                for year in re.findall(r'\b(?:19|20)\d{2}\b', str(literal.this))}

    # An aggregate FILTER affects only that aggregate, never its neighbours.
    where_years = set().union(*(predicate_years(where) for where in wheres
                                if not isinstance(where.parent, exp.Filter)))
    conditional_years = set().union(*(predicate_years(case.args.get('this'))
                                      for case in tree.find_all(exp.If)
                                      if case.args.get('this') is not None))
    filter_years = set().union(*(predicate_years(where) for where in wheres
                                 if isinstance(where.parent, exp.Filter)))
    if tables.intersection({'sales', 'dim_month'}) and years:
        if not years <= where_years | conditional_years | filter_years:
            return ('Restrict every requested reporting year using invoice_date on sales or invoice_month on dim_month. '
                    'A selected year label is not a date filter; keep all requested periods in WHERE/IN or CASE conditions.')
        if not years <= where_years:
            # Without an outer date scope, every aggregate of a measure needs a
            # conditional date scope. Do not let one CASE launder an all-time SUM.
            for aggregate in tree.find_all(exp.AggFunc):
                measures = [col for col in aggregate.find_all(exp.Column)
                            if col.name.lower() in {'revenue', 'net_revenue', 'gross_revenue', 'quantity', 'invoice_no'}]
                if not measures:
                    continue
                local_filter = (predicate_years(aggregate.parent.args['expression'])
                                if isinstance(aggregate.parent, exp.Filter) else set())
                def conditional_measure_scoped(column):
                    # Each measure must sit in a date-restricted THEN branch.
                    # ELSE revenue would reintroduce rows outside the period.
                    current = column
                    while current is not aggregate and current.parent is not None:
                        parent = current.parent
                        if (isinstance(parent, exp.If) and parent.args.get('true') is current
                                and predicate_years(parent.this) & years):
                            return True
                        current = parent
                    return False
                if not (local_filter & years or all(conditional_measure_scoped(col) for col in measures)):
                    return ('An aggregate still covers all dates. Put the requested dates in WHERE, '
                            'use FILTER for that aggregate, or restrict every measured value with '
                            'CASE WHEN date condition THEN measure ELSE 0/NULL END.')

    for predicate in tree.find_all(exp.Predicate):
        if any(col.name.lower() == 'country' for col in predicate.find_all(exp.Column)):
            if any(lit.is_string and str(lit.this).strip().lower() in {'uk', 'u.k.', 'great britain'}
                   for lit in predicate.find_all(exp.Literal)):
                return "Use country = 'United Kingdom' (or IN/<> with that full name); the database has no country named 'UK'."

    explicit_exclusion = bool(re.search(r'(?:exclud\w*|without)\s+(?:the\s+)?cancellations?|cancellations?\s+(?:are\s+)?excluded', text))
    revenue_question = 'revenue' in text and (explicit_exclusion or not re.search(r'cancel|return', text))
    product_scope = bool(re.search(r'product|stock code|item', text))
    customer_ranking = bool(re.search(r'(?:top|which).*customers', text))
    identified = bool(re.search(r'customer_id IS NOT NULL|NOT customer_id IS NULL', normalized, re.I))
    grouped_customers = any(col.name.lower() == 'customer_id' for group in tree.find_all(exp.Group)
                            for col in group.find_all(exp.Column))
    def requires_boolean(predicate, column, value, negated=False):
        # This tests implication: can a matching row have the opposite flag?
        # Under NOT, De Morgan swaps AND/OR. Inverting only the desired value
        # would wrongly accept NOT(is_cancellation AND country = ...).
        if isinstance(predicate, exp.Paren):
            return requires_boolean(predicate.this, column, value, negated)
        if isinstance(predicate, exp.Not):
            return requires_boolean(predicate.this, column, value, not negated)
        if isinstance(predicate, (exp.And, exp.Or)):
            combine = all if (isinstance(predicate, exp.Or) != negated) else any
            return combine(requires_boolean(part, column, value, negated)
                           for part in (predicate.this, predicate.expression))
        if isinstance(predicate, exp.Column):
            return predicate.name.lower() == column and value is not negated
        if isinstance(predicate, (exp.EQ, exp.Is)):
            left, right = predicate.this, predicate.expression
            expected = not value if negated else value
            return ((isinstance(left, exp.Column) and left.name.lower() == column
                     and isinstance(right, exp.Boolean) and right.this is expected)
                    or (isinstance(right, exp.Column) and right.name.lower() == column
                        and isinstance(left, exp.Boolean) and left.this is expected))
        return False

    cancellation_excluded = any(requires_boolean(where.this, 'is_cancellation', False) for where in wheres)
    extra_exclusions = bool(re.search(r'is_product|is_outlier|quantity', normalized)) and not product_scope

    if years and tables.intersection({'dim_customer', 'dim_product'}) and 'sales' not in tables:
        return 'dim_customer and dim_product cover all dates. Query sales with the requested year filter; do not filter first_order or last_order to estimate annual totals.'
    if revenue_question and 'dim_customer' in tables and 'sales' not in tables:
        return 'dim_customer.net_revenue includes cancellations. Query SUM(revenue) from sales WHERE NOT is_cancellation for customer revenue.'
    if 'sales' in tables and customer_ranking and (not identified or not grouped_customers):
        return ('Customer rankings require customer_id IS NOT NULL and GROUP BY customer_id on sales. '
                'Group by customer_id, not country; keep the requested date and revenue/order metric.')
    if 'sales' in tables and revenue_question and 'dim_customer' in tables and 'customer' not in text:
        return ('For overall/country revenue use sales directly: country is already present. '
                'A customer join can drop guest purchases or repeat customer totals; remove that join.')
    if re.search(r'\borders?\b', text) and 'sales' in tables:
        distinct_orders = any(isinstance(count.this, exp.Distinct)
                              and any(col.name.lower() == 'invoice_no' for col in count.find_all(exp.Column))
                              for count in tree.find_all(exp.Count))
        if not distinct_orders or not cancellation_excluded or extra_exclusions:
            return ('Completed orders require COUNT(DISTINCT invoice_no) and WHERE NOT is_cancellation. '
                    'For customer rankings also require customer_id IS NOT NULL. '
                    'Remove product, outlier and quantity exclusions unless the question asks for that population. '
                    'Keep the requested customer/date filters.')
    if revenue_question and 'sales' in tables:
        if not cancellation_excluded:
            return 'Revenue must exclude cancellations: add WHERE NOT is_cancellation (or AND NOT is_cancellation to the existing WHERE).'
        if extra_exclusions:
            return ('Overall/country/customer revenue must not use product-only, outlier or quantity exclusions. '
                    'Remove is_product, is_outlier and quantity filters; keep NOT is_cancellation and the requested country/customer/date filters.')
    if revenue_question and 'dim_month' in tables:
        if any(col.name.lower() == 'gross_revenue' for col in tree.find_all(exp.Column)):
            return 'Use dim_month.net_revenue for revenue excluding cancellations; gross_revenue includes cancellations.'
        monthly_rank = bool(re.search(r'month.*(?:highest|lowest)|(?:highest|lowest).*month', text))
        if monthly_rank and not any(requires_boolean(where.this, 'is_complete_month', True) for where in wheres):
            return 'Monthly revenue rankings require WHERE is_complete_month. Select invoice_month, net_revenue and trading_days; include all tied winners.'
    return None


def run_sql(sql, log, question=None):
    """Run a bounded SELECT, close on errors, and never silently truncate evidence."""
    from threading import Timer
    problem = check_sql(sql)
    if not problem and question:
        problem = query_scope_problem(sql, question)
    if problem:
        return add_to_log(log, "run_sql", sql, False, error=problem)
    con = None
    timer = None
    try:
        con = duckdb.connect(DB, read_only=True, config={"enable_external_access": "false"})
        # A valid SELECT can still be expensive. Interrupt runaway model queries.
        timer = Timer(20, con.interrupt)
        timer.start()
        cursor = con.execute(sql)
        columns = [d[0] for d in cursor.description]
        if len(columns) != len(set(columns)):
            raise ValueError("Give every result column a unique alias.")
        result = cursor.fetchmany(51)
        if len(result) > 50:
            raise ValueError("More than 50 rows. Aggregate the result or add an explicit LIMIT 50.")
        rows = [{c: json_safe(v) for c, v in zip(columns, row)} for row in result]
        return add_to_log(log, "run_sql", sql, True, rows)
    except Exception as error:
        return add_to_log(log, "run_sql", sql, False, error=str(error))
    finally:
        if timer:
            timer.cancel()
            timer.join()
        if con:
            con.close()


# The model does NOT get to write Python. It picks an operation from this
# list and gives us the numbers; we do the arithmetic ourselves.
#
# This is stricter than letting it write code, and it is also easier to check:
# the operation and its inputs are recorded in the log, so Tier 3 can redo the
# sum without trusting anything the model said.

OPERATIONS = ["pct_change", "difference", "ratio", "share", "sum", "mean"]


def calculate(operation, values):
    """Do the arithmetic. Returns (result, how_we_worked_it_out)."""
    if operation == "pct_change":
        if len(values) != 2:
            raise ValueError("pct_change needs [new, old]")
        new, old = values
        if old == 0:
            raise ValueError("cannot work out a percentage change from zero")
        return (new - old) / old * 100, "(%s - %s) / %s * 100" % (new, old, old)

    if operation == "difference":
        if len(values) != 2:
            raise ValueError("difference needs [a, b]")
        return values[0] - values[1], "%s - %s" % (values[0], values[1])

    if operation == "ratio":
        if len(values) != 2 or values[1] == 0:
            raise ValueError("ratio needs [top, bottom] and bottom cannot be zero")
        return values[0] / values[1], "%s / %s" % (values[0], values[1])

    if operation == "share":
        if len(values) != 2 or values[1] == 0:
            raise ValueError("share needs [part, total] and total cannot be zero")
        return values[0] / values[1] * 100, "%s / %s * 100" % (values[0], values[1])

    if operation == "sum":
        if not values:
            raise ValueError("sum needs at least one number")
        return sum(values), " + ".join(str(v) for v in values)

    if operation == "mean":
        if not values:
            raise ValueError("mean needs at least one number")
        return sum(values) / len(values), "mean of %d numbers" % len(values)

    raise ValueError("unknown operation: %s" % operation)


def run_python(request, log):
    """Calculate only with finite numbers already returned by successful tools."""
    import math
    try:
        if not isinstance(request, dict):
            raise ValueError("the calculation request must be an object")
        operation = request.get("operation")
        raw = request.get("values")
        if not isinstance(raw, list) or any(isinstance(v, bool) for v in raw):
            raise ValueError("values must be a list of numbers")
        values = [float(v) for v in raw]
        known = numbers_in_log(log)
        if not all(math.isfinite(v) and any(math.isclose(v, n, rel_tol=1e-6, abs_tol=.01)
                                            for _, n in known) for v in values):
            raise ValueError("every input must come from an earlier successful query or calculation")
        result, code = calculate(operation, values)
        if not math.isfinite(result):
            raise ValueError("the result must be finite")
        unit = request.get("unit") or ("%" if operation in ("pct_change", "share") else None)
        return add_to_log(log, "run_python", code, True,
                          [{"result_name": request.get("result_name") or operation,
                            "value": round(result, 4), "unit": unit,
                            "operation": operation, "inputs": values}])
    except (TypeError, ValueError, OverflowError) as error:
        return add_to_log(log, "run_python", str(request), False, error=str(error))


CHART_TYPES = ["bar", "line", "scatter", "pie", "table", "none"]


def make_chart(spec, log):
    """Validate a real source; infer missing axes only when the choice is clear."""
    problem = None
    spec = dict(spec) if isinstance(spec, dict) else {}
    if spec.get('type') not in CHART_TYPES:
        problem = 'choose a supported chart type'
    elif spec['type'] != 'none':
        source = next((c for c in log if c.get('n') == spec.get('source_tool_call')
                       and c.get('ok') and c.get('tool') == 'run_sql'), None)
        if not source or not source.get('rows'):
            problem = 'chart source must be a successful non-empty SQL result'
        else:
            rows = source['rows']
            numeric = [k for k in rows[0] if all(isinstance(r.get(k), (int,float)) and not isinstance(r.get(k),bool) for r in rows)]
            labels = [k for k in ('invoice_month', 'description', 'country', 'stock_code') if k in rows[0]]
            if not spec.get('x') and labels:
                spec['x'] = labels[0]
            if not spec.get('y') and len(numeric) == 1:
                spec['y'] = numeric[0]
            if any(spec.get(axis) not in rows[0] for axis in ('x','y')):
                problem = 'chart axes must be columns in the source result'
            elif spec['type'] != 'table' and spec['y'] not in numeric:
                problem = 'the chart y-axis must contain numerical measurements'
            elif spec.get('x') == spec.get('y'):
                problem = 'choose distinct chart axes'
            elif spec['type'] == 'pie' and (len(rows) < 2 or any(float(r[spec['y']]) < 0 for r in rows) or sum(float(r[spec['y']]) for r in rows) <= 0):
                problem = 'A share chart needs at least two non-negative categories with a positive total.'
    return add_to_log(log, 'make_chart', json.dumps(spec), not problem,
                      [spec] if not problem else [], error=problem)


def model_usage(responses):
    """Keep actual Ollama token counts; missing usage is unknown, not zero."""
    if not responses:
        return {'model_calls': 0, 'input_tokens': 0, 'output_tokens': 0}
    return {key: sum(r[key] for r in responses) if all(r.get(key) is not None for r in responses) else None
            for key in ('model_calls', 'input_tokens', 'output_tokens')}


def ask_gemma(system, question, shape):
    """Ask the model for JSON of a particular shape.

    `shape` is a JSON schema. Ollama forces the reply to match it, which is a
    lot more reliable than asking nicely and hoping. If the reply still will
    not parse we try once more with a blunter instruction.
    """
    import ollama

    usage = []
    for attempt in range(2):
        prompt = question if attempt == 0 else question + """

YOUR LAST REPLY WAS NOT VALID JSON.
Reply with ONE complete JSON object matching the schema. Keep the text short.
Do not write anything outside the JSON object."""

        try:
            reply = ollama.Client(timeout=120).chat(model=MODEL, format=shape,
                                options={"temperature": 0, "num_predict": 2048},
                                messages=[{"role": "system", "content": system},
                                          {"role": "user", "content": prompt}])
        except ollama.ResponseError as error:
            if error.status_code != 500:
                raise  # Missing models/authentication need an actionable setup error.
            usage.append({'model_calls': 1, 'input_tokens': None, 'output_tokens': None})
            # The caller can retry planning/SQL within its normal budget. An
            # optional chart failure must not discard a successful data query.
            return {'broken_json': True, 'error': 'Model generation failed: ' + str(error),
                    '_usage': model_usage(usage)}
        usage.append({'model_calls': 1, 'input_tokens': reply.get('prompt_eval_count'),
                      'output_tokens': reply.get('eval_count')})
        try:
            parsed = json.loads(reply["message"]["content"])
            if isinstance(parsed, dict):
                parsed['_usage'] = model_usage(usage)
                return parsed
        except Exception:
            continue

    return {"broken_json": True, "_usage": model_usage(usage)}


# The shapes we ask the model to fill in. Keeping them small is deliberate -
# a 4B model fills in three fields reliably and fifteen fields badly.

PLAN_SHAPE = {
    "type": "object",
    "properties": {
        "sufficient_data": {"type": "boolean"},
        "reason": {"type": "string"},
        "steps": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "step": {"type": "integer"},
                "tool": {"type": "string", "enum": ["run_sql", "run_python", "make_chart"]},
                "objective": {"type": "string"},
            },
            "required": ["step", "tool", "objective"]}},
    },
    "required": ["sufficient_data", "steps"],
}

SQL_SHAPE = {"type": "object",
             "properties": {"sql": {"type": "string"}},
             "required": ["sql"]}

PYTHON_SHAPE = {
    "type": "object",
    "properties": {
        "operation": {"type": "string", "enum": OPERATIONS},
        "values": {"type": "array", "items": {"type": "number"}},
        "result_name": {"type": "string"},
        "unit": {"type": "string"},
    },
    "required": ["operation", "values", "result_name"],
}

CHART_SHAPE = {
    "type": "object",
    "properties": {
        "type": {"type": "string", "enum": CHART_TYPES},
        "source_tool_call": {"type": "integer"},
        "x": {"type": "string"},
        "y": {"type": "string"},
        "title": {"type": "string"},
    },
    "required": ["type", "source_tool_call", "x", "y", "title"],
}

ANSWER_SHAPE = {
    "type": "object",
    "properties": {
        "findings": {"type": "string"},
        "claims": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "text": {"type": "string"},
                "value": {"type": "number"},
                "kind": {"type": "string", "enum": ["number", "boolean"]},
                "row": {"type": "integer"},
                "column": {"type": "string"},
                "unit": {"type": "string"},
                "from_call": {"type": "integer"},
                "calc": {"type": "string",
                         "enum": ["none", "pct_change", "share", "sum", "diff",
                                  "difference", "ratio", "mean"]},
                "inputs": {"type": "array", "items": {"type": "number"}},
            },
            "required": ["text", "value", "unit", "from_call", "calc", "inputs"]}},
        "kpis": {"type": "object"},
        "limitations": {"type": "string"},
        "insufficient_data": {"type": "boolean"},
    },
    "required": ["findings", "claims"],
}


SCHEMA = """
Database: a UK online gift wholesaler, Dec 2009 to 9 Dec 2011.

TABLE sales (one row per invoice line, not one row per order)
  invoice_no, stock_code, description, quantity, unit_price, customer_id,
  country, revenue (= quantity * unit_price), invoice_date, invoice_month,
  is_cancellation, is_product, is_outlier
  invoice_date is a timestamp. invoice_month is text 'YYYY-MM'.
  Country names are stored in full: UK is 'United Kingdom'.
  sales already contains customer_id, country and product details: no join is
  needed for customer or country rankings. NULL customer_id means a guest.

TABLE dim_month (one row per calendar month)
  invoice_month TEXT 'YYYY-MM', trading_days, net_revenue, gross_revenue,
  is_complete_month BOOLEAN
  net_revenue EXCLUDES cancellations. gross_revenue INCLUDES cancellations.
  Use net_revenue for ordinary monthly revenue and divide by trading_days
  for revenue per trading day. Do not apply sales-only fields to this table.

TABLE dim_product stock_code, description, units_sold, gross_revenue
TABLE dim_customer customer_id, country, first_order, last_order, orders, net_revenue
  These two dimension summaries cover ALL dates; they cannot answer a year's
  totals. dim_customer.net_revenue includes cancellations despite its name.
  Use sales for revenue, order and customer analyses.

There is NO cost, profit, margin, discount, competitor or customer age data.
"""

RULES = """
BUSINESS RULES
1. Overall, country and customer revenue: SUM(revenue) FROM sales WHERE
   NOT is_cancellation. Keep every remaining line: do NOT add is_product,
   is_outlier or quantity filters. Keep guests for overall/country revenue.
2. ONLY product-specific revenue questions add is_product AND NOT is_outlier.
   Group by stock_code; use mode(description) for the name. Units sold needs
   quantity > 0; do not apply that condition to revenue or completed orders.
3. Completed orders = COUNT(DISTINCT invoice_no), NOT COUNT(*) or line counts,
   with NOT is_cancellation. Customer rankings also need customer_id IS NOT NULL
   and GROUP BY customer_id. Customer IDs are labels, not measurements.
4. For monthly totals, rankings, comparisons and trading days use dim_month.
   Select invoice_month as a label and trading_days as context. Complete-month
   revenue rankings require WHERE is_complete_month. Monthly growth is
   100.0 * (new - old) / NULLIF(old,0); use unrounded inputs until the final ROUND.
   Per-day growth first divides EACH month's net_revenue by its trading_days.
5. December 2011 is incomplete; never compare it as a full month.
6. A share needs a part and the WHOLE population: do not restrict the denominator
   to the chosen country. Use country = 'United Kingdom' for UK.
7. For an extreme value, return every tied winner unless a fixed list size is
   requested. Use a maximum/minimum subquery or DENSE_RANK; LIMIT 1 hides ties.
8. A year applies to every sales query for that question. Use year(invoice_date)
   or timestamp bounds; dim_month uses 'YYYY-MM' text bounds/IN. Each SELECT
   must use the schema of its own table. Prefer one simple grouped query over
   unnecessary joins. Never join sales to monthly totals and then SUM those totals.
9. Never say what CAUSED something, only what contributed to it. If the data
   cannot answer the question, explain what is missing instead of guessing.
"""

EXAMPLES = """
SQL PATTERNS (adapt the country, period, metric and limit to the QUESTION)

Country revenue ranking, no product filters or customer joins:
  SELECT country, ROUND(SUM(revenue),2) AS revenue FROM sales
  WHERE NOT is_cancellation AND year(invoice_date) = 2010
  GROUP BY country ORDER BY revenue DESC LIMIT 3

Identified customer order ranking, one row per customer:
  SELECT customer_id, COUNT(DISTINCT invoice_no) AS orders FROM sales
  WHERE NOT is_cancellation AND customer_id IS NOT NULL
    AND year(invoice_date) = 2010
  GROUP BY customer_id ORDER BY orders DESC LIMIT 3

A country's share with an unrestricted denominator:
  SELECT SUM(CASE WHEN country = 'Sweden' THEN revenue ELSE 0 END) AS country_revenue,
         SUM(revenue) AS total_revenue,
         100.0 * SUM(CASE WHEN country = 'Sweden' THEN revenue ELSE 0 END)
           / NULLIF(SUM(revenue),0) AS share_pct
  FROM sales WHERE NOT is_cancellation AND year(invoice_date) = 2010

Monthly comparison evidence (then calculate the requested change):
  SELECT invoice_month, net_revenue AS revenue, trading_days,
         net_revenue / NULLIF(trading_days,0) AS revenue_per_day, is_complete_month
  FROM dim_month WHERE invoice_month IN ('2010-05','2010-06')
  ORDER BY invoice_month

All tied extrema, with the same scope in the inner and outer query:
  SELECT invoice_month, trading_days FROM dim_month
  WHERE trading_days = (SELECT MIN(trading_days) FROM dim_month)
  ORDER BY invoice_month
"""

SYSTEM = "You are a careful business data analyst.\n" + SCHEMA + RULES

PLAN_PROMPT = SYSTEM + """
Plan how to answer the question. Do NOT answer it - you have not seen any data.

List the steps in order. Each step uses one tool:
  run_sql      fetch numbers from the database
  run_python   work something out from numbers a query already returned
  make_chart   show the result

Use the shortest useful plan. A total usually needs only one SQL query.
Do not add arithmetic when SQL already returned the answer. Only add a chart
when the question asks for one or when comparing several categories or periods.
Set sufficient_data to false if the data cannot answer the question at all.
"""

ANSWER_PROMPT = SYSTEM + """
You asked for some queries and here are the results. Write the answer using
ONLY these numbers.

Every number in findings must also appear as a claim, with:
  from_call  which call it came from
  calc       none, unless you worked it out - then pct_change/share/sum/diff/ratio
  inputs     the numbers you worked it out from

Write plain business prose in findings. Do not include call IDs, calculation
labels, or input lists in the prose; those belong in claim fields.
State that revenue excludes cancellations. Use digits for numerical claims.
For yes/no facts, leave numerical claims empty; the tool evidence supplies
typed boolean facts. Never encode a boolean as 0 or 1.
For numeric facts include row and column when copied from SQL. Keep every
product and its value in a separate claim. Preserve product names exactly.
Do not say what caused anything. Say what contributed.
"""


def sql_prompt(question, objective, feedback=''):
    """The caller supplies SYSTEM once; keep user-side SQL guidance focused."""
    return (EXAMPLES
            + '\nQUESTION\n' + question
            + '\nRELEVANT RULES\n' + query_guidance(question)
            + ('\nPREVIOUS ATTEMPT FEEDBACK — correct these issues\n' + str(feedback) if feedback else '')
            + '\nWRITE ONE SELECT QUERY FOR THIS STEP ONLY\n' + str(objective)
            + '\nUse exact schema columns, unique output aliases and the requested population. Return SQL in the required JSON field.')


def repair_prompt(question, objective, sql, error, feedback=''):
    """Give a small model one concrete edit, without repeating schema/examples."""
    message = str(error)
    normalized = re.sub(r'\b\w+\.', '', str(sql).lower())
    normalized = re.sub(r'\s+', ' ', normalized)
    instructions = []
    if 'Customer rankings require' in message:
        if not re.search(r'customer_id is not null|not customer_id is null', normalized):
            instructions.append('Add AND customer_id IS NOT NULL to the existing WHERE clause, before GROUP BY.')
        try:
            tree = sqlglot.parse_one(str(sql), read='duckdb')
            grouped = any(col.name.lower() == 'customer_id' for group in tree.find_all(exp.Group)
                          for col in group.find_all(exp.Column))
        except Exception:
            grouped = False
        if not grouped:
            instructions.append('Select customer_id and group by customer_id; do not group a customer ranking by country.')
        instructions.append('Keep the existing requested year, metric, ordering and limit.')
    elif 'Completed orders require' in message:
        instructions.append('Use COUNT(DISTINCT invoice_no) for the orders result. Keep NOT is_cancellation. Remove quantity, is_product and is_outlier filters.')
    elif 'must not use product-only' in message:
        instructions.append('Delete is_product, is_outlier and quantity conditions from WHERE. Keep NOT is_cancellation and the requested date/country/customer conditions.')
    elif "database has no country named 'UK'" in message:
        instructions.append("Replace the country literal 'UK' with 'United Kingdom'; preserve the rest of the query.")
    elif 'Complete' in message or 'complete_month' in message:
        instructions.append('Add is_complete_month to the dim_month WHERE clause. Keep the requested months, revenue metric and ordering.')
    else:
        instructions.append('Correct the error below. Preserve the requested dates, labels and metrics; remove rejected filters. Use only columns belonging to the named table.')
    return ('QUESTION\n' + str(question)
            + '\nSTEP\n' + str(objective)
            + '\nFAILED SQL\n' + str(sql)
            + '\nERROR\n' + message
            + ('\nVERIFIER FEEDBACK\n' + str(feedback) if feedback else '')
            + '\nREQUIRED EDIT\n' + '\n'.join(instructions)
            + '\nReturn one corrected SELECT in the required JSON sql field. Apply the edit; do not repeat the failed query unchanged.')


def python_prompt(question, objective, log):
    return (SYSTEM
            + "\nQUESTION\n" + question
            + "\n\nSTEP\n" + objective
            + "\n\nNUMBERS THE QUERIES RETURNED\n" + show_results(log)
            + "\nPick the operation and give us the numbers. We do the arithmetic.")


def chart_prompt(question, objective, log):
    return (SYSTEM
            + "\nQUESTION\n" + question
            + "\n\nSTEP\n" + objective
            + "\n\nRESULTS SO FAR\n" + show_results(log)
            + "\nChoose a chart. source_tool_call is the call number to plot.")


def answer_prompt(question, log, feedback=""):
    text = ("QUESTION\n" + question
            + "\n\nTOOL RESULTS\n" + show_results(log))
    if feedback:
        text += "\n\nYOUR LAST ANSWER FAILED CHECKING\n" + feedback
    return text


def touches_incomplete_month(log):
    """Did any query return a month flagged as incomplete?"""
    for call in log:
        if not call["ok"]:
            continue
        for row in call["rows"]:
            if row.get("is_complete_month") is False:
                return True
    return False


def mentions_december_2011(question):
    text = question.lower()
    return ("december 2011" in text or "dec 2011" in text
            or "2011-12" in text)


def claims_from_log(log):
    """Preserve typed facts and their row labels when the writer omits claims."""
    claims = []
    for call in log:
        if not call.get('ok') or call.get('tool') == 'make_chart':
            continue
        for index, row in enumerate(call['rows']):
            if call['tool'] == 'run_python':
                claims.append({'text': str(row.get('result_name', 'Calculation')) + ': ' + str(row['value']),
                               'value': row['value'], 'unit': row.get('unit'), 'from_call': call['n'],
                               'calc': row['operation'], 'inputs': row['inputs']})
                continue
            label = str(row.get('description') or row.get('country') or row.get('invoice_month') or '')
            for key, value in row.items():
                if key in ('customer_id', 'stock_code') or not isinstance(value, (int,float,bool)):
                    continue
                if isinstance(value, bool):
                    text = ('The month is complete.' if value else 'The month is incomplete.') if key == 'is_complete_month' else key + (' is true.' if value else ' is false.')
                    claims.append(evidence_claim(call, index, key, text))
                    continue
                unit = {'revenue':'GBP','net_revenue':'GBP','aov':'GBP','trading_days':'days','units':'units'}.get(key,'')
                text = (label + ': ' if label else '') + key.replace('_',' ') + ' = ' + str(value)
                claims.append(evidence_claim(call, index, key, text, unit))
    return claims


def data_gap_reason(question):
    """Cheap scope checks for known absent data; keep the weak planner from guessing."""
    text = question.lower()
    missing = r"\b(profit|profits|margin|margins|cost|costs|cogs|discounts?|promotions?|campaigns?|marketing|demographics?|competitors?|customer age|web traffic|website traffic)\b"
    if re.search(missing, text):
        return "This dataset contains sales transactions, but not the cost, promotion or other external data needed to answer this question."
    if re.search(r"\b(forecast|predict|prediction|next quarter|next year|next month)\b", text):
        return "Forecasting is outside this project's scope. These historical transactions cannot establish future revenue."
    comparison = re.search(r"\b(compare|compared|comparison|versus|vs|beat|change|growth|decline|increase|decrease|difference|drop)\b", text)
    normalized = "per trading day" in text or "per day" in text
    if mentions_december_2011(question) and comparison and not normalized:
        return "December 2011 is an incomplete reporting month. A full-month comparison would be misleading; ask for individual totals or revenue per trading day."
    return None


def question_contract(question):
    """Compile a small, explicit vocabulary of questions into KPI-safe queries.

    Match the whole question so an extra country/product filter is never lost.
    These are reusable query patterns, not benchmark answers: dates and limits
    come from the question, and every value still comes from the live database.
    Unrecognised wording continues through the model-driven tool loop.
    """
    import calendar
    text = re.sub(r"\s+", " ", question.lower()).strip().rstrip("?.")
    year = r"(?P<year>(?:19|20)\d{2})"
    count = r"(?P<count>\d+|one|two|three|four|five|six|seven|eight|nine|ten)"
    names = {name.lower(): i for i, name in enumerate(calendar.month_name) if name}
    month = "(?:" + "|".join(names) + ")"
    result = None
    patterns = [
        (rf"(?:what was (?:our |the )?(?:total )?revenue|(?:total )?revenue) in {year}", "revenue"),
        (rf"how many customers (?:bought from us|purchased) in {year}", "customers"),
        (rf"how many orders did we (?:take|receive) in {year}", "orders"),
        (rf"how many units did we sell in {year}", "units"),
        (rf"what was (?:our |the )?average order value in {year}", "aov"),
        (rf"what was (?:our |the )?cancellation rate in {year}", "cancellation_rate"),
        (rf"(?:list the {count} products that generated the most revenue in {year},? with the revenue for each)", "products"),
        (rf"top {count} products (?:by revenue )?in {year}", "products"),
        (rf"which {count} products generated the most revenue in {year}", "products"),
        (rf"which {count} products sold the most units in {year}", "product_units"),
    ]
    for pattern, kind in patterns:
        match = re.fullmatch(pattern, text)
        if not match:
            continue
        y = match['year']
        where = f"NOT is_cancellation AND year(invoice_date) = {y}"
        if kind == 'cancellation_rate':
            where = f"year(invoice_date) = {y}"
        expressions = {'cancellation_rate': 'ROUND(100.0 * COUNT(DISTINCT CASE WHEN is_cancellation THEN invoice_no END) / NULLIF(COUNT(DISTINCT CASE WHEN NOT is_cancellation THEN invoice_no END),0),2)',
                       'revenue': 'ROUND(SUM(revenue),2)',
                       'customers': 'COUNT(DISTINCT customer_id)',
                       'orders': 'COUNT(DISTINCT invoice_no)',
                       'units': 'SUM(quantity)',
                       'aov': 'ROUND(SUM(revenue)/NULLIF(COUNT(DISTINCT invoice_no),0),2)'}
        if kind in ('products', 'product_units'):
            words = 'zero one two three four five six seven eight nine ten'.split()
            raw = match['count']
            n = int(raw) if raw.isdigit() else words.index(raw)
            if not 1 <= n <= 50:
                return None
            metric = 'revenue' if kind == 'products' else 'units'
            if metric == 'units':
                where += ' AND quantity > 0'
            sql = (f"SELECT stock_code, mode(description) AS description, {expressions[metric]} AS {metric} "
                   f"FROM sales WHERE {where} AND is_product AND NOT is_outlier "
                   f"GROUP BY stock_code ORDER BY {metric} DESC, stock_code LIMIT {n}")
            result = {'kind': kind, 'metric': metric, 'count': n, 'year': y, 'sql': sql}
        else:
            if kind == 'customers':
                where += ' AND customer_id IS NOT NULL'
            if kind == 'units':
                where += ' AND quantity > 0'
            result = {'kind': kind, 'metric': kind, 'year': y,
                      'sql': f"SELECT {expressions[kind]} AS {kind} FROM sales WHERE {where}"}
        break
    if result:
        return result
    foreign = re.fullmatch(rf"which country outside (?:the )?(?:uk|united kingdom) generated the most revenue in {year},? and how much was it", text)
    if foreign:
        return {'kind': 'country_rank', 'metric': 'revenue', 'year': foreign['year'],
                'sql': "SELECT country, ROUND(SUM(revenue),2) AS revenue FROM sales WHERE NOT is_cancellation "
                       f"AND year(invoice_date) = {foreign['year']} AND country <> 'United Kingdom' "
                       "GROUP BY country ORDER BY revenue DESC, country LIMIT 1"}
    match = re.fullmatch(rf"is (?P<month>{month}) {year} (?:a )?complete month(?: in the dataset)?", text)
    if match:
        period = f"{match['year']}-{names[match['month']]:02d}"
        return {'kind': 'completeness', 'period': period,
                'sql': f"SELECT invoice_month, trading_days, is_complete_month FROM dim_month WHERE invoice_month = '{period}'"}
    match = re.fullmatch(rf"did (?P<new>{month}) (?P<newyear>(?:19|20)\d{{2}}) beat (?P<old>{month}) (?P<oldyear>(?:19|20)\d{{2}}) on revenue,? and by what percentage", text)
    if match:
        new = f"{match['newyear']}-{names[match['new']]:02d}"
        old = f"{match['oldyear']}-{names[match['old']]:02d}"
        if new == old:
            return None
        return {'kind': 'comparison', 'new': new, 'old': old,
                'sql': "SELECT invoice_month, ROUND(net_revenue,2) AS revenue, trading_days, is_complete_month "
                       f"FROM dim_month WHERE invoice_month IN ('{old}', '{new}') ORDER BY invoice_month"}
    return None


def evidence_claim(call, row_index, column, text, unit=''):
    """Bind a fact to a specific cell, keeping labels and measurements together."""
    value = call['rows'][row_index][column]
    return {'text': text, 'value': value, 'unit': unit, 'from_call': call['n'],
            'calc': 'none', 'inputs': [], 'row': row_index, 'column': column,
            'kind': 'boolean' if isinstance(value, bool) else 'number'}


def patterned_answer(question, contract, log):
    """Execute common analyses without making a small model copy or invent facts.

    Shared by Tier 2 and Tier 3. Only Tier 3 applies independent verification.
    Keep the query, calculation and chart calls in the same evidence log as the
    model-driven path, so this optimisation stays visible in the evaluation.
    """
    start = time.time()
    call = run_sql(contract['sql'], log, question)
    fields, filters = sql_metadata(contract['sql'])
    answer = {'question': question, 'tier': 2, 'findings': '', 'claims': [], 'kpis': {},
              'fields_used': fields, 'filters_used': filters, 'chart': None,
              'insufficient_data': False, 'limitations': '', 'log': log, 'retries': 0,
              'plan': ['run_sql: use the matching KPI query pattern'], 'seconds': 0,
              'route': 'query_pattern', 'scope_check': 'matching KPI query pattern', 'usage': model_usage([])}
    rows = call['rows']
    if not call['ok'] or not rows or any(v is None for r in rows for v in r.values()):
        answer.update(findings='No usable data was returned for this request.',
                      limitations=call.get('error') or 'The requested period has no data.')
        return answer
    claims = answer['claims']
    kind = contract['kind']
    spec = None
    if kind in ('products', 'product_units', 'country_rank'):
        metric = contract['metric']
        for i, row in enumerate(rows):
            amount = f"£{row[metric]:,.2f}" if metric == 'revenue' else f"{row[metric]:,} units"
            claims.append(evidence_claim(call, i, metric, f"{row.get('description', row.get('country'))}: {amount}.",
                                         'GBP' if metric == 'revenue' else 'units'))
        spec = {'type': 'bar', 'source_tool_call': call['n'], 'x': 'country' if kind == 'country_rank' else 'description',
                'y': metric, 'title': 'Leading market' if kind == 'country_rank' else 'Leading products'}
        answer['limitations'] = ('Cancellations and the United Kingdom are excluded.' if kind == 'country_rank' else 'Cancellations, non-product lines and flagged outliers are excluded.')
    elif kind == 'completeness':
        row = rows[0]
        label = 'complete' if row['is_complete_month'] else 'incomplete'
        claims.append(evidence_claim(call, 0, 'is_complete_month', f"The month is {label}."))
        claims.append(evidence_claim(call, 0, 'trading_days', f"It contains {row['trading_days']} trading days.", 'days'))
        answer['limitations'] = 'An incomplete month cannot be compared as a full month.' if not row['is_complete_month'] else ''
    elif kind == 'comparison':
        by_month = {r['invoice_month']: i for i, r in enumerate(rows)}
        if set(by_month) != {contract['new'], contract['old']}:
            answer.update(findings='Both requested months must have data before they can be compared.',
                          limitations='One or more reporting months are missing.')
            return answer
        if not all(r['is_complete_month'] for r in rows):
            answer.update(findings='A requested month is incomplete, so a full-month comparison would be misleading.',
                          insufficient_data=True, limitations='Ask for revenue per trading day instead.')
            return answer
        new, old = (rows[by_month[contract[k]]]['revenue'] for k in ('new', 'old'))
        calculation = run_python({'operation': 'pct_change', 'values': [new, old], 'unit': '%',
                                  'result_name': 'Revenue change'}, log)
        for i, row in enumerate(rows):
            claims.append(evidence_claim(call, i, 'revenue', f"In {row['invoice_month']}, revenue was £{row['revenue']:,.2f}.", 'GBP'))
            claims.append(evidence_claim(call, i, 'trading_days', f"In {row['invoice_month']}, there were {row['trading_days']} trading days.", 'days'))
        if calculation['ok']:
            value = calculation['rows'][0]['value']
            text = ('Yes, revenue increased' if value > 0 else 'No, revenue decreased' if value < 0 else 'No, revenue was unchanged')
            claims.append({'text': f"{text} by {abs(value):.2f}%." if value >= 0 else f"No, revenue changed by {value:.2f}%.",
                           'value': value, 'unit': '%', 'from_call': calculation['n'], 'calc': 'pct_change',
                           'inputs': [new, old], 'input_cells': [{'from_call': call['n'], 'row': by_month[contract[k]], 'column': 'revenue'} for k in ('new', 'old')]})
        else:
            answer['limitations'] = 'Percentage change is undefined when the earlier revenue is zero. '
        answer['limitations'] += 'Revenue excludes cancellations. Trading-day counts are shown for context.'
        spec = {'type': 'line', 'source_tool_call': call['n'], 'x': 'invoice_month', 'y': 'revenue', 'title': 'Monthly revenue'}
    else:
        metric = contract['metric']
        value = rows[0][metric]
        if metric == 'cancellation_rate':
            text = f"Cancellation invoices were {value:.2f}% of completed orders."
            unit = '%'
        elif metric in ('revenue', 'aov'):
            text = f"{'Revenue' if metric == 'revenue' else 'Average order value'} was £{value:,.2f}."
            unit = 'GBP'
        else:
            text = f"There were {value:,} {'identified customers' if metric == 'customers' else metric}."
            unit = metric
        claims.append(evidence_claim(call, 0, metric, text, unit))
        answer['limitations'] = 'Cancellations are excluded.'
        if metric == 'cancellation_rate':
            answer['limitations'] = 'Cancellation invoices cannot be linked to their original orders. This is a ratio of cancellations to completed orders in the same period, not the share of orders later cancelled.'
        if metric == 'customers':
            answer['limitations'] += ' Transactions without a customer ID are excluded from this customer count.'
    if spec:
        chart_call = make_chart(spec, log)
        if chart_call['ok']:
            answer['chart'] = chart_call['rows'][0]
    answer['findings'] = ' '.join(c['text'] for c in claims)
    if kind not in ('completeness', 'cancellation_rate'):
        answer['findings'] += ' Cancellations are excluded.'
    answer['seconds'] = round(time.time() - start, 2)
    return answer


def label_spans(text, answer):
    """Only exact categorical labels in successful SQL evidence exempt digits.

    Never exempt numeric strings, arbitrary text columns or all small numbers.
    That would allow an unsupported count to slip through the verifier.
    """
    spans = []
    for call in answer.get('log', []):
        if call.get('ok') and call.get('tool') == 'run_sql':
            # A filtered identifier is also a label when explicitly introduced
            # as 'customer' or 'stock code'; it is not an asserted quantity.
            try:
                tree = sqlglot.parse_one(call.get('code', ''), read='duckdb')
                for where in tree.find_all(exp.Where):
                    for predicate in where.find_all(exp.EQ):
                        column, literal = predicate.this, predicate.expression
                        if isinstance(column, exp.Column) and isinstance(literal, exp.Literal):
                            prefix = {'customer_id': r'customer(?:[ _]id)?', 'stock_code': r'(?:stock|product) code'}.get(column.name.lower())
                            if prefix:
                                pattern = r'\b' + prefix + r'\s+' + re.escape(str(literal.this)) + r'(?!\w)'
                                spans += [(m.start(),m.end()) for m in re.finditer(pattern,text,re.I)]
            except Exception:
                pass  # Malformed SQL never creates a label exemption.
            for row in call.get('rows', []):
                for key in ('description', 'stock_code', 'customer_id', 'country', 'invoice_month'):
                    label = str(row.get(key) or '')
                    # DuckDB stores nullable customer IDs as doubles. Rendered
                    # integer IDs retain their identity, not a new quantity.
                    if key == 'customer_id' and re.fullmatch(r'\d+\.0+', label):
                        label = label.split('.')[0]
                    if not label or (key not in ('stock_code', 'customer_id') and not re.search(r'[A-Za-z-]', label)):
                        continue
                    # A numerical code needs an explicit identifier prefix.
                    pattern = re.escape(label)
                    if label.isdigit():
                        if key == 'customer_id':
                            pattern += r'(?:\.0+)?'
                        pattern = (r'customer(?:[ _]id)?\s+' if key == 'customer_id' else r'(?:stock code\s+|product code\s+)') + pattern
                    spans += [(m.start(), m.end()) for m in re.finditer(r'(?<!\w)' + pattern + r'(?!\w)', text, re.I)]
    return spans


def is_evidence_label(text, start, end, answer):
    return any(left <= start and end <= right for left, right in label_spans(text, answer))


def claim_cell(claim, log):
    """Resolve an explicit cell reference without Python's False == 0 shortcut."""
    source = next((c for c in log if c.get('n') == claim.get('from_call') and c.get('ok') and c.get('tool') == 'run_sql'), None)
    index, column = claim.get('row'), claim.get('column')
    if not source or type(index) is not int or not 0 <= index < len(source.get('rows', [])):
        return False, None
    row = source['rows'][index]
    return (True, row[column]) if column in row else (False, None)


def tier2(question, log=None, feedback=""):
    """Plan, run the steps, then write the answer from the real rows.

    The model never sees the database and never writes Python. It says what it
    wants; we run it and record what happened.
    """
    log = log if log is not None else []
    start = time.time()
    first_call = len(log)
    usage = []
    def model_call(system, prompt, shape):
        response = ask_gemma(system, prompt, shape)
        usage.append(response.get('_usage', {}))
        return response

    def stop(findings, limitations=None, insufficient=False, plan_steps=None):
        return {"question": question, "tier": 2, "findings": findings,
                "claims": [], "kpis": {}, "fields_used": [], "filters_used": [],
                "chart": None, "insufficient_data": insufficient,
                "limitations": limitations, "log": log, "retries": 0,
                "plan": plan_steps or [], "route": "model_tools",
                "failure": None if insufficient else (limitations or findings), "usage": model_usage(usage), "seconds": round(time.time() - start, 1)}

    gap = data_gap_reason(question)
    if gap:
        answer = stop(gap, gap, True)
        answer["route"] = "rules_refusal"
        return answer

    contract = question_contract(question)
    if contract:
        return patterned_answer(question, contract, log)

    # ---- 1. plan
    ask = question if not feedback else question + "\n\nLast attempt failed:\n" + feedback
    plan = model_call(PLAN_PROMPT, ask, PLAN_SHAPE)

    if plan.get("broken_json"):
        return stop("The planner did not return valid JSON.",
                    plan.get("error") or "the model did not return valid JSON")

    steps = [s for s in (plan.get("steps") or []) if isinstance(s, dict)]
    steps = sorted(steps, key=lambda s: s.get("step") if isinstance(s.get("step"), (int, float)) else 0)
    plan_text = ["%s: %s" % (s.get("tool"), s.get("objective")) for s in steps]

    # A small model often sets sufficient_data to false while still listing
    # steps it wants to run. gemma3:4b did exactly that on "what was our total
    # revenue in 2011", and Tier 2 refused a question it could easily answer.
    #
    # So we treat the STEPS as the real signal. If it asked for queries, it
    # thinks it can answer. We only refuse when it says no AND gives us
    # nothing to run.
    says_no = not plan.get("sufficient_data", True)

    if says_no and not steps:
        reason = plan.get("reason") or "The data cannot answer this question."
        return stop(reason, reason, True, plan_text)

    if not steps:
        return stop("The planner produced no steps to run.",
                    "the planner marked the question answerable but gave no steps",
                    False, plan_text)

    # ---- 2. run the steps
    chart = None

    for step in steps[:6]:
        tool = step.get("tool")
        objective = step.get("objective", "")

        if tool == "run_sql":
            request = model_call(SYSTEM, sql_prompt(question, objective, feedback), SQL_SHAPE)
            call = run_sql(request.get("sql", ""), log, question)

            # one repair attempt, with the error message
            if not call["ok"]:
                fix = model_call(SYSTEM,
                                repair_prompt(question, objective,
                                              call["code"], call["error"], feedback),
                                SQL_SHAPE)
                call = run_sql(fix.get("sql", ""), log, question)

            if not call["ok"]:
                return stop("The query could not be made to run.",
                            call["error"], False, plan_text)

        elif tool == "run_python":
            # Do not work out a month-over-month change when one of the months
            # is incomplete. December 2011 has 8 trading days.
            if touches_incomplete_month(log) or mentions_december_2011(question):
                continue
            request = model_call(SYSTEM, python_prompt(question, objective, log),
                                PYTHON_SHAPE)
            run_python(request, log)

        elif tool == "make_chart":
            spec = model_call(SYSTEM, chart_prompt(question, objective, log),
                             CHART_SHAPE)
            call = make_chart(spec, log)
            if call["ok"] and spec.get("type") != "none":
                chart = call["rows"][0]

    # ---- 3. refuse if nothing worked
    useful = [c for c in log[first_call:]
              if c["ok"] and c["tool"] in ("run_sql", "run_python")]
    if not useful:
        # Nothing worked. Refusing is the honest outcome - we have no numbers,
        # so any answer would be invented.
        return stop("No query produced any evidence, so there is no answer to give.",
                    "no successful query or calculation", True, plan_text)

    # ---- 4. write the answer from the real rows
    reply = model_call(ANSWER_PROMPT, answer_prompt(question, log, feedback),
                      ANSWER_SHAPE)

    if reply.get("broken_json"):
        return stop("The model did not return a usable answer.",
                    reply.get("error") or "the model did not return valid JSON", False, plan_text)

    claims = reply.get("claims") or []
    if not isinstance(claims, list):
        claims = []
    claims = [c for c in claims if isinstance(c, dict)]
    if not claims:
        claims = claims_from_log(log[first_call:])

    # Boolean facts have a separate typed route; never coerce False to zero.
    for fact in claims_from_log(log[first_call:]):
        if fact.get('kind') == 'boolean':
            if not any(c.get('kind') == 'boolean' and c.get('from_call') == fact['from_call'] and c.get('column') == fact['column'] and c.get('row') == fact['row'] for c in claims):
                claims.append(fact)

    # ---- 5. fields and filters come from the SQL, not from the model
    fields, filters = [], []
    for call in log[first_call:]:
        if call["tool"] != "run_sql" or not call["ok"]:
            continue
        f, w = sql_metadata(call["code"])
        fields += [x for x in f if x not in fields]
        filters += [x for x in w if x not in filters]

    limitations = reply.get("limitations")
    if (touches_incomplete_month(log) or mentions_december_2011(question)) \
            and not limitations:
        limitations = "An incomplete month is not comparable with a complete month."

    return {"question": question, "tier": 2,
            "findings": reply.get("findings", ""),
            "claims": claims,
            "kpis": reply.get("kpis") or {},
            "fields_used": fields or [],
            "filters_used": filters or [],
            "chart": chart,
            "route": "model_tools", "scope_check": "numeric evidence only",
            "insufficient_data": bool(reply.get("insufficient_data")),
            "limitations": limitations,
            "log": log, "retries": 0, "plan": plan_text, "usage": model_usage(usage),
            "seconds": round(time.time() - start, 1)}



def tier1(question):
    """Same output contract, but no tools and no access to database results."""
    start = time.time()
    prompt = ("You are a business data analyst for a UK online gift wholesaler. "
              "Data covers December 2009 to December 2011. There is no cost, profit, "
              "marketing or demographic data. Answer the question with findings and "
              "numeric claims. Set insufficient_data if you cannot answer. You have no tools.")
    reply = ask_gemma(prompt, question, ANSWER_SHAPE)
    if reply.get("broken_json"):
        raise ValueError("Tier 1 did not return a usable JSON object")
    return {"question": question, "tier": 1, "findings": reply.get("findings", ""),
            "claims": reply.get("claims") or [], "kpis": {}, "fields_used": [],
            "filters_used": [], "chart": None, "plan": [],
            "route": "model", "scope_check": "numeric evidence only",
            "insufficient_data": bool(reply.get("insufficient_data")),
            "limitations": reply.get("limitations"), "log": [], "retries": 0, "usage": reply.get("_usage", {}),
            "seconds": round(time.time() - start, 1)}


def about_equal(a, b, tol=1e-6):
    """Allow rounding, but reject malformed, boolean and non-finite values."""
    import math
    try:
        if isinstance(a, bool) or isinstance(b, bool):
            return False
        a, b = float(a), float(b)
        return math.isfinite(a) and math.isfinite(b) and abs(a-b) <= max(.011, tol * max(abs(a), abs(b)))
    except (TypeError, ValueError, OverflowError):
        return False


def which_call(number, log_numbers):
    """Find the evidence call without assuming call IDs are list positions."""
    return next((n for n, value in log_numbers if about_equal(number, value)), None)


def as_numbers(values):
    """Normalize scalar/list/dict model inputs; ignore booleans and non-finite junk."""
    import math
    if isinstance(values, dict):
        values = list(values.values())
    elif not isinstance(values, (list, tuple)):
        values = [values]
    out = []
    for value in values:
        if isinstance(value, dict):
            out.extend(as_numbers(value))
        elif not isinstance(value, (bool, list, tuple)):
            try:
                number = float(str(value).replace(",", "").replace("GBP", "").replace("£", "").strip())
                if math.isfinite(number):
                    out.append(number)
            except (TypeError, ValueError, OverflowError):
                pass
    return out


def recalculate(calc, inputs):
    """Use the same arithmetic operations as the tools, including both difference names."""
    try:
        value, _ = calculate("difference" if calc == "diff" else calc, as_numbers(inputs))
        return value if about_equal(value, value) else None
    except (TypeError, ValueError, OverflowError, ZeroDivisionError):
        return None


def claim_context_problem(claim, log):
    """A copied number must match its cell, unit and categorical label.

    Legacy claims without cell coordinates are accepted only when their cited
    result has one unambiguous matching cell. Arithmetic has its own input checks.
    """
    if claim.get('calc') not in (None, 'none') or isinstance(claim.get('value'), bool):
        return None
    candidates = []
    for call in log:
        if not call.get('ok') or call.get('tool') != 'run_sql':
            continue
        if claim.get('from_call') is not None and claim['from_call'] != call['n']:
            continue
        for index, row in enumerate(call.get('rows', [])):
            for column, value in row.items():
                if about_equal(claim.get('value'), value):
                    if 'row' in claim and claim['row'] != index:
                        continue
                    if 'column' in claim and claim['column'] != column:
                        continue
                    candidates.append((call, index, column, row))
    # A direct claim can also cite a recalculated run_python result.
    if not candidates:
        return None
    if len(candidates) > 1:
        return 'The value occurs in multiple cells. Specify the exact row and column.'
    call, index, column, row = candidates[0]
    unit = str(claim.get('unit') or '').lower().strip()
    if column in ('revenue', 'net_revenue', 'aov') and unit not in ('gbp', '£', 'pounds'):
        return 'The cited monetary value must use GBP.'
    if column == 'trading_days' and unit not in ('days', 'trading days'):
        return 'The cited trading-day count must use days.'
    text = str(claim.get('text') or '').lower()
    for key in ('description', 'country'):
        if row.get(key) and str(row[key]).lower() not in text:
            if key == 'description' and str(row.get('stock_code', '')) in text:
                continue
            return 'Include the exact result label with this value: ' + str(row[key])
    claim.update(from_call=call['n'], row=index, column=column)
    return None


def verify(answer):
    """Check provenance and arithmetic independently; malformed claims fail closed."""
    log = answer.get("log") or []
    evidence = numbers_in_log(log)
    scope_errors = []
    if answer.get('question'):
        for call in log:
            if call.get('ok') and call.get('tool') == 'run_sql':
                try:
                    problem = query_scope_problem(call.get('code'), answer['question'])
                except Exception:
                    problem = 'The evidence query could not be checked.'
                if problem:
                    scope_errors.append(problem)
    answer['scope_errors'] = scope_errors
    calls = {c.get("n"): c for c in log if isinstance(c, dict)}
    raw_claims = answer.get("claims") or []
    if not isinstance(raw_claims, list):
        raw_claims = [raw_claims]
    answer["claims"] = [dict(c) if isinstance(c, dict) else {"text": str(c)} for c in raw_claims]
    for claim in answer["claims"]:
        claim.update(status="unsupported", evidence=None, problem=None)
        value = claim.get("value")
        calc = claim.get("calc") or "none"
        inputs = as_numbers(claim.get("inputs"))
        source = which_call(value, evidence)
        # Respect an explicit source. Do not silently reassign a wrong citation.
        cited = claim.get("from_call")
        if cited is not None and calc == "none":
            source = next((n for n, v in evidence if n == cited and about_equal(value, v)), None)
        context_problem = claim_context_problem(claim, log)
        has_cell = 'row' in claim or 'column' in claim
        found, cell_value = claim_cell(claim, log) if has_cell else (False, None)
        if scope_errors:
            claim['problem'] = 'Query meaning failed checking: ' + scope_errors[0]
        elif claim.get('kind') == 'boolean' or isinstance(value, bool):
            if (claim.get('kind') == 'boolean' and type(value) is bool and found
                    and type(cell_value) is bool and cell_value is value and calc == 'none'):
                claim.update(status='supported', evidence='Exact boolean SQL cell')
            else:
                claim['problem'] = 'A boolean fact needs an exact boolean SQL cell reference.'
        elif has_cell and (not found or not about_equal(value, cell_value)):
            claim['problem'] = 'The cited row and column do not contain this value.'
        elif claim.get('column') in ('revenue', 'net_revenue', 'aov') and claim.get('unit') != 'GBP':
            claim['problem'] = 'Revenue and order value use GBP.'
        elif claim.get('column') == 'trading_days' and claim.get('unit') not in ('days', 'trading days'):
            claim['problem'] = 'Trading-day evidence must use days.'
        elif context_problem:
            claim['problem'] = context_problem
        elif value is None or not about_equal(value, value):
            claim["problem"] = "a claim needs a finite numeric value"
        elif calc != "none":
            expected = recalculate(calc, inputs)
            input_sources = [which_call(x, evidence) for x in inputs]
            refs = claim.get('input_cells')
            bound_inputs = (refs is None or (isinstance(refs, list) and len(refs) == len(inputs)
                            and all(claim_cell(ref, log)[0] and about_equal(value, claim_cell(ref, log)[1])
                                    for value, ref in zip(inputs, refs))))
            if not bound_inputs:
                claim['problem'] = 'Calculation inputs do not match their cited cells.'
            elif expected is None:
                claim["problem"] = "unknown calculation or invalid inputs"
            elif not inputs or any(n is None for n in input_sources):
                claim["problem"] = "calculation inputs are not backed by query evidence"
            elif not about_equal(value, expected):
                claim["status"] = "flagged"
                claim["problem"] = "we get %.4f, not %s" % (expected, value)
            else:
                claim.update(status="supported", from_call=input_sources[0],
                             evidence="Recalculated %s from calls %s" % (calc, sorted(set(input_sources))))
        elif source is not None:
            claim.update(status="supported", from_call=source, evidence=calls[source].get("code"))
        else:
            claim["problem"] = "this number is not in the cited successful result"
    answer["reconciliation"] = check_totals(answer)
    answer["undeclared"] = undeclared_numbers(answer)
    return answer


def check_totals(answer):
    """Check explicit sums. A top-five share breakdown need not cover the whole market."""
    problems = []
    for claim in answer.get("claims", []):
        if claim.get("calc") == "sum":
            expected = recalculate("sum", claim.get("inputs"))
            if expected is not None and not about_equal(expected, claim.get("value")):
                problems.append("The stated sum disagrees with its own inputs.")
    # Completeness cannot be inferred from an arbitrary list of percentage claims.
    # Their individual part/total arithmetic is checked in verify().
    return problems


# Only explicit calendar context is exempt. Small counts and percentages need evidence.


def number_tokens(text):
    """Keep spans for exact redaction, including £ amounts, signs and scaled numbers."""
    pattern = r"(?<![\w.])[-+−]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?:[eE][+-]?\d+)?(?:\s*(?:billion|million|thousand|bn|[kKmM])\b)?"
    tokens = []
    for match in re.finditer(pattern, str(text or "")):
        token = match.group().replace(",", "").replace("−", "-")
        suffix = re.search(r"\s*(billion|million|thousand|bn|[kKmM])$", token)
        scale = 1
        if suffix:
            scale = {"billion": 1e9, "bn": 1e9, "million": 1e6, "m": 1e6,
                     "thousand": 1e3, "k": 1e3}[suffix.group(1).lower()]
            token = token[:suffix.start()]
        try:
            tokens.append((match.start(), match.end(), float(token) * scale))
        except (ValueError, OverflowError):
            pass
    return tokens


def is_date_context(text, start, end):
    """Recognise calendar wording while retaining genuine counts and amounts."""
    for date in re.finditer(r"\b(?:19|20)\d{2}-\d{2}(?:-\d{2})?\b", text):
        if date.start() <= start < end <= date.end():
            return True
    token = text[start:end]
    if not re.fullmatch(r"(?:19|20)\d{2}", token):
        return False
    prefix = text[max(0, start-25):start]
    suffix = text[end:end+40]
    named_year = bool(re.search(r"(?:in|during|for|year|january|february|march|april|may|june|july|august|september|october|november|december)\s*$", prefix, re.I))
    reporting_year = (bool(re.search(r"(?:our|the|its|their)\s*$", prefix, re.I))
                      and bool(re.match(r"\s+(?:revenue|sales|financial results|reporting period)\b", suffix, re.I)))
    return named_year or reporting_year


def find_numbers(text):
    """Extract digit-based numbers; number words need a separate semantic review."""
    return [value for _, _, value in number_tokens(text)]


def undeclared_numbers(answer):
    """Require numerical prose to have a claim; check small counts and percentages too."""
    declared = [c.get("value") for c in answer.get("claims", [])]
    texts = public_prose_texts(answer)
    texts += [c.get("text") for c in answer.get("claims", [])]
    out = []
    for text in texts:
        text = re.sub(r"\[call\s+\d+\]", "", str(text or ""), flags=re.I)
        for start, end, value in number_tokens(text):
            if not is_date_context(text, start, end) and not is_evidence_label(text, start, end, answer) and not any(about_equal(value, d) for d in declared):
                out.append(value)
    return out


def write_feedback(answer):
    """Give concrete repair feedback for execution, evidence and completeness."""
    lines = []
    execution = execution_feedback(answer)
    if execution:
        lines.append(execution)
    lines += ['%s: %s' % (c.get('text'), c.get('problem'))
              for c in answer.get('claims', []) if c.get('status') != 'supported']
    if answer.get('undeclared'):
        lines.append('Declare or remove these numbers: %s' % answer['undeclared'])
    lines += answer.get('scope_errors', [])
    lines += answer.get('reconciliation', [])
    lines += answer.get('completeness_issues', [])
    lines = list(dict.fromkeys(str(line) for line in lines if line))
    if not lines:
        return ''
    return ('Fix these issues using successful query evidence and explicit calculation inputs. '
            'Keep the original question and its filters.\n' + '\n'.join(lines))


def tier3(question, max_retries=2):
    """Repair failures, keep complete supported attempts, and render checked facts."""
    from copy import deepcopy
    start = time.time()
    feedback = ''
    usage, history = [], []
    best = None
    best_quality = None
    selected = 0
    for attempt in range(max_retries + 1):
        # Each attempt has fresh evidence. History is measured but never used
        # to substantiate claims in a different attempt.
        answer = verify(recover_evidence_claims(verify(tier2(question, [], feedback))))
        answer.setdefault('question', question)
        answer['completeness_issues'] = completeness_problems(answer)
        usage.append(answer.get('usage', {}))
        answer['retries'] = attempt
        feedback = write_feedback(answer)
        supported = len(claims_to_show(answer))
        quality = (not bool(answer.get('scope_errors')),
                   not bool(execution_feedback(answer) or answer['completeness_issues']),
                   supported, -len(answer.get('undeclared', [])),
                   -len(answer.get('claims', [])) + supported)
        history.append({'attempt': attempt, 'answer': deepcopy(answer), 'feedback': feedback})
        if best_quality is None or quality > best_quality:
            best, best_quality, selected = deepcopy(answer), quality, attempt
        if not feedback:
            break
    answer = best
    answer['usage'] = model_usage(usage)
    answer['attempts'] = history
    answer['selected_attempt'] = selected
    answer['retries'] = attempt
    answer['tier'] = 3
    answer['seconds'] = round(time.time() - start, 2)
    answer['kpis'] = {}
    answer = render_verified_answer(answer)
    visible = claims_to_show(answer)
    # Plots are public output: they must agree with the selected attempt's
    # supported cells, not just happen to contain the same numeric values.
    chart = answer.get('chart')
    if chart:
        source = next((c for c in answer.get('log', []) if c.get('n') == chart.get('source_tool_call')
                       and c.get('ok') and c.get('tool') == 'run_sql'), None)
        chart_ok = bool(source and not answer.get('scope_errors'))
        if chart_ok:
            chart_ok = all(any(claim.get('from_call') == source['n']
                               and claim.get('row') == index
                               and claim.get('column') == chart.get('y')
                               and about_equal(claim.get('value'), row.get(chart.get('y')))
                               for claim in visible) for index, row in enumerate(source['rows']))
        if not chart_ok:
            answer['chart'] = None
            answer['limitations'] = (answer.get('limitations') or '') + ' The chart was withheld because its values could not be verified.'
    hidden = sum(c.get('status') != 'supported' for c in answer.get('claims', []))
    if hidden or answer.get('undeclared'):
        answer['limitations'] = (answer.get('limitations') or '') + ' Some figures could not be verified and were withheld.'
    if answer.get('completeness_issues'):
        answer['limitations'] = (answer.get('limitations') or '') + ' ' + public_completeness_note(answer)
    return answer


# =====================================================================
# 5. THE VERIFIER  (this is our contribution - no AI in here)
# =====================================================================


def claims_to_show(answer):
    """Display supported answer facts; keep supported context in the audit data."""
    return [claim for claim in answer.get('claims', [])
            if claim.get('status') == 'supported' and claim.get('display_role') != 'context']


def clean_findings(answer):
    """Redact whole number spans, so hiding 12 cannot accidentally alter 1,200."""
    text = re.sub(r"\[call\s+\d+\]", "", str(answer.get("findings") or ""), flags=re.I).strip()
    text = re.sub(r'\bcustomer_id(?=\s+\d)', 'Customer', text, flags=re.I)
    supported = [c.get("value") for c in claims_to_show(answer)]
    for start, end, value in reversed(number_tokens(text)):
        if not is_date_context(text, start, end) and not is_evidence_label(text, start, end, answer) and not any(about_equal(value, v) for v in supported):
            text = text[:start] + "[unverified]" + text[end:]
    return text


def score(answer):
    """Retained-answer evidence plus SQL-only execution across available attempts."""
    claims = answer.get('claims', [])
    supported = sum(c.get('status') == 'supported' for c in claims)
    flagged = sum(c.get('status') == 'flagged' for c in claims)
    attempts = answer.get('attempts') or []
    calls = ([c for attempt in attempts for c in attempt.get('answer', {}).get('log', [])]
             if attempts else answer.get('log', []))
    sql_calls = [c for c in calls if c.get('tool') == 'run_sql']
    successful = sum(bool(c.get('ok')) for c in sql_calls)
    return {'claims': len(claims), 'supported': supported, 'flagged': flagged,
            'unsupported': len(claims) - supported, 'undeclared': len(answer.get('undeclared', [])),
            'evidence_coverage': supported / len(claims) if claims else None,
            'unsupported_rate': (len(claims) - supported) / len(claims) if claims else None,
            'queries': len(sql_calls), 'sql_successes': successful,
            'queries_ok': successful / len(sql_calls) if sql_calls else None,
            'query_history_complete': bool(attempts) or not answer.get('retries')}


def show(answer):
    """Show the direct answer and its meaning before the detailed evidence."""
    icons = {'supported': '[OK]  ', 'flagged': '[WARN]', 'unsupported': '[HIDE]'}
    print(clean_findings(answer))
    if answer.get('interpretation'):
        print('\nWhat this means:', answer['interpretation'])
    if answer.get('suggested_next_steps'):
        print('\nUseful next steps:')
        for suggestion in answer['suggested_next_steps']:
            print('- ' + str(suggestion))
    if claims_to_show(answer):
        print('\nDetailed figures:')
    for claim in claims_to_show(answer):
        print(icons.get(claim.get('status'), '      '), claim.get('text'))
        if claim.get('problem'):
            print('         !', claim['problem'])
        elif claim.get('evidence'):
            print('         evidence:', claim['evidence'].strip().replace('\n', ' ')[:70])
    hidden = sum(claim.get('status') != 'supported' for claim in answer.get('claims', []))
    if hidden:
        print('\n%d claim(s) hidden - could not be verified.' % hidden)
    if answer.get('undeclared'):
        print('%d number(s) in the summary were never declared as claims.' % len(answer['undeclared']))
    if answer.get('limitations'):
        print('\nLimitations:', answer['limitations'])
    if answer.get('insufficient_data'):
        print('\nThe data cannot answer this question.')
    print('\ntier %s | %s retries | %s seconds' % (answer.get('tier'), answer.get('retries'), answer.get('seconds')))


def execution_feedback(answer):
    """Repair an unsuccessful analysis even when it contains no numeric claims."""
    if answer.get('route') in ('query_pattern', 'rules_refusal'):
        return ''
    calls = answer.get('log') or []
    successful = [c for c in calls if c.get('ok') and c.get('tool') in ('run_sql', 'run_python')]
    failure = answer.get('failure')
    if successful and not failure:
        return ''
    failed = [c for c in calls if not c.get('ok') and c.get('tool') in ('run_sql', 'run_python')]
    messages = []
    if failed:
        # Keep the most recent error and the actual attempted query together.
        last = failed[-1]
        messages.append('The previous analysis failed. Do not repeat this query unchanged:\n'
                        + str(last.get('code', '')) + '\nReason: ' + str(last.get('error', 'Unknown tool error')))
    elif failure:
        messages.append('The previous analysis failed: ' + str(failure))
    elif not successful and not answer.get('claims'):
        messages.append('No usable evidence was produced. Plan a read-only query that returns the requested measurements and labels.')
    return '\n'.join(messages)


def tier2_with_retries(question, max_retries=2):
    """Optional equal-budget control: execution feedback only, no verification."""
    from copy import deepcopy
    start = time.time()
    history, usage = [], []
    feedback = ''
    for attempt in range(max_retries + 1):
        answer = tier2(question, [], feedback)
        usage.append(answer.get('usage', {}))
        feedback = execution_feedback(answer)
        history.append({'attempt': attempt, 'answer': deepcopy(answer), 'feedback': feedback})
        if not feedback:
            break
    answer['attempts'] = history
    answer['selected_attempt'] = attempt
    answer['retries'] = attempt
    answer['usage'] = model_usage(usage)
    answer['seconds'] = round(time.time() - start, 2)
    answer['control'] = 'execution_retries_without_verification'
    return answer



def query_guidance(question):
    """Highlight the relevant business rules without selecting SQL or answers."""
    text = (question or '').lower()
    tips = []
    if 'revenue' in text and not re.search(r'product|stock code|item', text):
        tips.append('Use the overall revenue population: exclude cancellations only; no product, outlier or quantity filters.')
    if re.search(r'\bcountr|\buk\b|united kingdom', text):
        tips.append("Country is already on sales. Use 'United Kingdom' for UK; keep guest purchases. Avoid customer joins.")
    if re.search(r'\bcustomers?\b', text):
        tips.append('Use sales, not all-date dim_customer totals. Group rankings by customer_id and exclude NULL customer_id.')
    if re.search(r'\borders?\b', text):
        tips.append('Count DISTINCT invoice_no; exclude cancellations and keep non-product lines.')
    if re.search(r'\bshare\b|percentage.*(?:uk|countr)|(?:uk|countr).*percentage', text):
        tips.append('Return both numerator and denominator; the denominator must include all requested countries, not just the numerator country.')
    if re.search(r'month|trading day|january|february|march|april|may|june|july|august|september|october|november|december', text):
        tips.append('Use dim_month. Include each requested period, revenue, trading_days and is_complete_month; use no sales-only filters.')
        if 'per trading day' in text or 'per day' in text:
            tips.append('Compute revenue / trading_days for EACH period, then compare these daily values as well as total revenues.')
        if re.search(r'highest|lowest|most|fewest|maximum|minimum', text):
            tips.append('Return every tied extreme. For revenue rankings use only complete months; for trading-day rankings include all months unless restricted.')
    return '\n'.join('- ' + tip for tip in tips)


def fact_cell(claim, log):
    """Find one exact SQL cell; model-written labels are never a source."""
    matches = []
    for call in log:
        if not call.get('ok') or call.get('tool') != 'run_sql':
            continue
        if claim.get('from_call') is not None and call.get('n') != claim['from_call']:
            continue
        for index, row in enumerate(call.get('rows') or []):
            for column, value in row.items():
                if 'row' in claim and index != claim['row']:
                    continue
                if 'column' in claim and column != claim['column']:
                    continue
                same = (type(value) is bool and type(claim.get('value')) is bool
                        and value is claim['value']) if isinstance(value, bool) else about_equal(value, claim.get('value'))
                if same:
                    matches.append((call, index, column, row))
    return matches[0] if len(matches) == 1 else None

def fact_label(row, call=None):
    """Keep names, identifiers and periods attached to their measured values."""
    import datetime
    labels = []
    for key in ('description', 'country', 'customer_id', 'stock_code', 'invoice_month', 'month', 'period'):
        value = row.get(key)
        if value is None or (key == 'stock_code' and row.get('description')):
            continue
        if key == 'customer_id':
            value = 'Customer ' + (str(int(value)) if isinstance(value, (int, float)) else str(value))
        elif key == 'stock_code':
            value = 'Stock code ' + str(value)
        elif key in ('invoice_month', 'month', 'period'):
            try:
                value = datetime.datetime.strptime(str(value)[:7], '%Y-%m').strftime('%B %Y')
            except ValueError:
                pass
        labels.append(str(value))
    # Scalar aggregates often keep the customer/country in WHERE, not SELECT.
    if not labels and call:
        try:
            tree = sqlglot.parse_one(call.get('code') or '', read='duckdb')
            for where in tree.find_all(exp.Where):
                for node in where.find_all(exp.EQ):
                    left, right = node.this, node.expression
                    if isinstance(left, exp.Column) and isinstance(right, exp.Literal):
                        if left.name in ('country', 'customer_id', 'stock_code', 'invoice_month'):
                            labels.append(fact_label({left.name: right.this}))
        except Exception:
            pass
    return ' · '.join(dict.fromkeys(label for label in labels if label))

def fact_metric(column, call=None):
    """Translate known result columns; unknown aliases remain neutral labels."""
    invoice_measure = invoice_count_measure(column, call)
    if invoice_measure:
        return invoice_measure.capitalize()
    names = {'aov': 'Average order value', 'net_revenue': 'Revenue',
             'customers': 'Identified customers',
             'trading_days': 'Trading days', 'cancellation_rate': 'Cancellation-to-completed-order ratio',
             'is_complete_month': 'Complete reporting month'}
    return names.get(column, str(column).replace('_', ' ').strip().capitalize())

def fact_value(value, column='', unit=''):
    """Format verified measurements consistently; never round identifiers here."""
    if isinstance(value, bool):
        return 'Yes' if value else 'No'
    if not about_equal(value, value):
        return ''
    column = str(column).lower()
    money = ('revenue' in column and not any(word in column for word in ('share', 'change', 'growth', 'pct', 'percent')))
    if money or column in ('aov', 'average_order_value') or unit == 'GBP':
        return f'£{float(value):,.2f}'
    if unit == '%' or any(word in column for word in ('pct', 'percent', 'cancellation_rate')):
        return f'{float(value):,.2f}%'
    return f'{float(value):,.0f}' if float(value).is_integer() else f'{float(value):,.2f}'

def recover_evidence_claims(answer):
    """Recover omitted tool facts using exact cells, never prose or reference answers.

    This is deliberately conservative: identifiers, calendar numbers and ranking
    metadata are not measurements. Existing claims remain available for auditing.
    """
    log = answer.get('log') or []
    answer['claims'] = [dict(claim) if isinstance(claim, dict) else {'text': str(claim)}
                        for claim in (answer.get('claims') or [])]
    identities = ('customer_id', 'stock_code', 'invoice_no', 'year', 'month', 'invoice_year', 'rank', 'row_number')
    for call in log:
        if not call.get('ok') or call.get('tool') != 'run_sql':
            continue
        try:
            if answer.get('question') and query_scope_problem(call.get('code'), answer['question']):
                continue
        except Exception:
            continue
        for index, row in enumerate(call.get('rows') or []):
            for column, value in row.items():
                measured = bool(re.search(r'revenue|orders?|invoices?|customers?|units?|quantity|trading_days|aov|share|percentage|pct|rate|total|count|amount|value|is_complete_month', column))
                if column in identities or not measured or not isinstance(value, (int, float, bool)):
                    continue
                if not isinstance(value, bool) and not about_equal(value, value):
                    continue
                if any(c.get('from_call') == call['n'] and c.get('row') == index and c.get('column') == column
                       for c in answer['claims']):
                    continue
                unit = ('days' if column == 'trading_days' else 'GBP' if column in ('revenue', 'net_revenue', 'aov') else '')
                text = (fact_label(row, call) + ': ' if fact_label(row, call) else '') + fact_metric(column, call) + ' = ' + fact_value(value, column)
                claim = evidence_claim(call, index, column, text, unit)
                claim['recovered_from_evidence'] = True
                answer['claims'].append(claim)
    return answer

def verified_claim_text(claim, answer):
    """Render meaning from a bound SQL cell or a checked arithmetic operation."""
    if claim.get('status') != 'supported':
        return ''
    log = answer.get('log') or []
    operation = claim.get('calc') or 'none'
    if operation == 'none':
        cell = fact_cell(claim, log)
        if not cell:
            # Some writers cite the calculation tool as a direct claim. Its
            # checked operation and inputs still give us safe wording.
            for call in log:
                if call.get('ok') and call.get('tool') == 'run_python' and call.get('n') == claim.get('from_call'):
                    for row in call.get('rows') or []:
                        if about_equal(row.get('value'), claim.get('value')) and row.get('operation'):
                            derived = dict(claim, calc=row['operation'], inputs=row.get('inputs'))
                            text = verified_claim_text(derived, answer)
                            claim['unit'] = derived.get('unit', '')
                            return text
            return ''
        call, index, column, row = cell
        # Dashboard captions also expose claim.unit, independently of prose.
        # Replace that model field even when the original text was harmless.
        invoice_measure = invoice_count_measure(column, call)
        claim['unit'] = ('invoices' if invoice_measure in ('cancellation invoices', 'invoices') else
                         'count' if invoice_measure == 'count' else verified_measurement_unit(column))
        label = fact_label(row, call)
        prefix = label + ': ' if label else ''
        if column == 'is_complete_month':
            return prefix + ('Yes, this is a complete reporting month.' if claim['value'] else 'No, this reporting month is incomplete.')
        if column == 'trading_days':
            return prefix + fact_value(claim['value']) + ' trading days.'
        return prefix + fact_metric(column, call) + ': ' + fact_value(claim['value'], column) + '.'
    inputs = as_numbers(claim.get('inputs'))
    labels = []
    input_columns = []
    input_periods = []
    for i, value in enumerate(inputs):
        refs = claim.get('input_cells') or []
        reference = dict(refs[i], value=value) if i < len(refs) and isinstance(refs[i], dict) else {'value': value}
        cell = fact_cell(reference, log)
        labels.append(fact_label(cell[3], cell[0]) if cell else '')
        input_columns.append(cell[2] if cell else '')
        input_periods.append(str(cell[3].get('invoice_month') or cell[3].get('month') or '')[:7] if cell else '')
    names = {'pct_change': 'Percentage change', 'share': 'Share', 'difference': 'Difference',
             'diff': 'Difference', 'sum': 'Sum', 'mean': 'Average', 'ratio': 'Ratio'}
    name = names.get(operation, 'Calculated result')
    if input_columns and all(is_daily_metric(column) for column in input_columns):
        name = 'Per-trading-day ' + name.lower()
    input_units = [verified_measurement_unit(column) for column in input_columns]
    consistent_unit = input_units[0] if input_units and len(set(input_units)) == 1 else ''
    unit = '%' if operation in ('pct_change', 'share') else consistent_unit if operation in ('difference', 'diff', 'sum', 'mean') else ''
    claim['unit'] = unit
    text = name + ': ' + fact_value(claim['value'], unit='GBP' if unit.startswith('GBP') else unit)
    if operation in ('pct_change', 'difference', 'diff') and len(labels) == 2 and all(labels):
        text += ' (' + labels[0] + ' compared with ' + labels[1] + ')'
        # A mathematically correct reversed calculation cannot answer the
        # question's Yes/No direction. Only emit that shortcut when the exact
        # input periods match the explicit 'Did A beat B?' ordering.
        requested = requested_periods(answer.get('question', ''))
        if (operation == 'pct_change' and re.search(r'^\s*did\b.*\bbeat\b', answer.get('question', ''), re.I)
                and len(requested) == 2 and input_periods == requested and consistent_unit):
            text = ('Yes. ' if claim['value'] > 0 else 'No. ') + text
    return text + '.'

def render_verified_answer(answer):
    """Render verified facts, then explain their meaning without model prose."""
    answer = focus_verified_extrema(answer)
    sentences = []
    for claim in claims_to_show(answer):
        text = verified_claim_text(claim, answer)
        claim['text'] = text
        if text and text not in sentences:
            sentences.append(text)
    gap = data_gap_reason(answer.get('question', ''))
    queries = [call for call in answer.get('log', []) if call.get('tool') == 'run_sql']
    if sentences:
        findings = ' '.join(sentences)
    elif gap:
        findings = gap
    elif queries and all(call.get('ok') and not any(
            isinstance(value, bool) or (isinstance(value, (int, float)) and about_equal(value, value))
            for row in call.get('rows', []) for value in row.values()) for call in queries):
        findings = 'No usable data was returned for this request.'
    else:
        findings = 'I could not verify the reported figures. Please try a more specific question.'
    answer['findings'] = findings
    answer['limitations'] = verified_limitations(answer)
    answer = apply_human_narrative(answer)
    if 'Cancellations are excluded.' in answer['limitations']:
        answer['findings'] += ' Cancellations are excluded.'
    return answer

def verified_limitations(answer):
    """Only display caveats backed by known data rules or the successful query."""
    notes = []
    calls = [call for call in answer.get('log', []) if call.get('tool') == 'run_sql' and call.get('ok')]
    sql = ' '.join(str(call.get('code', '')).lower() for call in calls)
    columns = {key for call in calls for row in call.get('rows', []) for key in row}
    if 'cancellation_rate' in columns:
        notes.append('Cancellation invoices cannot be linked to their original orders. This is a ratio of cancellations to completed orders in the same period, not the share of orders later cancelled.')
    elif re.search(r'\bnot\s+(?:\w+\.)?is_cancellation\b', sql) or ('dim_month' in sql and 'net_revenue' in sql):
        notes.append('Cancellations are excluded.')
    if re.search(r'\bcustomer_id\s+is\s+not\s+null\b', sql):
        notes.append('Transactions without a customer ID are excluded from customer counts and rankings.')
    if re.search(r'\bnot\s+(?:\w+\.)?is_outlier\b', sql):
        notes.append('Flagged outliers are excluded.')
    if any(row.get('is_complete_month') is False for call in calls for row in call.get('rows', [])):
        notes.append('An incomplete reporting month cannot be compared as a full month.')
    return ' '.join(notes)

def completeness_problems(answer):
    """Check observable omissions; never consult benchmark answers or guess data.

    A short result can be legitimate if the database has fewer groups. We only
    request more rows when SQL itself has a too-small limit. Tie checks target
    explicit plural/tie requests and discrete trading-day extrema, rather than
    rejecting every ordinary singular peak-revenue query.
    """
    question = str(answer.get('question') or '').lower()
    if data_gap_reason(question):
        return []
    calls = [call for call in answer.get('log', []) if call.get('tool') == 'run_sql' and call.get('ok') and call.get('rows')]
    claims = claims_to_show(answer)
    problems = []
    if calls and re.search(r'\bshare\b|\bproportion\b|\b(?:percent|percentage)\s+(?:of|came\s+from|comes\s+from)\b', question):
        if not any(c.get('calc') == 'share' or re.search(r'share|percent|pct|proportion|fraction', str(c.get('column', '')), re.I) for c in claims):
            problems.append('The question requests a share, but the answer only reports source amounts. Include the verified part / total * 100 result, binding the part and total cells, or return an explicit share column from SQL.')
    wants_amount_difference = bool(re.search(r'by how much|\bdifference\b|how much (?:more|less|higher|lower)', question)) and bool(re.search(r'\bpounds?\b|\bgbp\b|£|\bmore revenue than\b|\bless revenue than\b', question))
    if calls and wants_amount_difference:
        if not any(c.get('calc') in ('difference', 'diff') or re.search(r'difference|\bdiff\b|gap|change_amount', str(c.get('column', '')), re.I) for c in claims):
            problems.append('The requested difference in pounds is missing. Subtract the second compared revenue from the first using the verified values and exact input_cells, or return an explicit difference column from SQL.')
    words = {'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10}
    match = re.search(r'\b(?:top|first|which)\s+(\d+|' + '|'.join(words) + r')\b', question)
    required = (int(match[1]) if match and match[1].isdigit() else words.get(match[1]) if match else None)
    ranked = [call for call in calls if any(key in row for row in call['rows'] for key in ('country', 'description', 'stock_code', 'customer_id'))]
    if required and ranked:
        call = max(ranked, key=lambda c: len(c['rows']))
        try:
            tree = sqlglot.parse_one(call['code'], read='duckdb')
            limit = tree.args.get('limit')
            limited_to = int(limit.expression.this) if limit and isinstance(limit.expression, exp.Literal) else None
        except Exception:
            limited_to = None
        if limited_to is not None and limited_to < required:
            problems.append(f'The question requests {required} ranked items, but SQL limits the result to {limited_to}. Return the requested number, or all available groups if fewer exist.')
        for index, row in enumerate(call['rows'][:required]):
            if not any(c.get('from_call') == call['n'] and c.get('row') == index for c in claims):
                problems.append('Include a bound measurement claim for every requested result row; the answer omits ' + (fact_label(row, call) or 'a result row') + '.')
    requested = requested_periods(question)
    comparing = bool(re.search(r'\b(compare|compared|versus|vs|beat|change|growth|decline|increase|decrease|difference|drop|dropped)\b', question))
    if comparing and len(requested) >= 2 and calls:
        observed = set()
        for claim in claims:
            cell = fact_cell(claim, answer.get('log') or []) if claim.get('calc') in (None, 'none') else None
            if cell:
                for key in ('invoice_month', 'month', 'period'):
                    if key in cell[3]:
                        observed.add(str(cell[3][key])[:7])
        if any(period not in observed for period in requested):
            problems.append('The comparison needs a verified measurement for each requested month. Return invoice_month with its measurement and bind both result cells.')
        if re.search(r'percent|percentage|\bpct\b|%', question) and not any(c.get('calc') == 'pct_change' or any(term in str(c.get('column', '')) for term in ('percent', 'pct', 'change')) for c in claims):
            problems.append('The requested percentage change is missing. Calculate (new - old) / old * 100 using the two verified values, with exact input_cells; a zero base makes percentage change undefined.')
        if 'per trading day' in question or 'per day' in question:
            daily_periods = set()
            daily_change = False
            for claim in claims:
                cell = fact_cell(claim, answer.get('log') or []) if claim.get('calc') in (None, 'none') else None
                if cell and is_daily_metric(cell[2]):
                    for key in ('invoice_month', 'month', 'period'):
                        if key in cell[3]:
                            daily_periods.add(str(cell[3][key])[:7])
                    daily_change |= any(term in cell[2] for term in ('change', 'pct', 'percent', 'difference'))
                if claim.get('calc') in ('pct_change', 'difference', 'diff'):
                    refs = claim.get('input_cells') or []
                    if refs and all(is_daily_metric(ref.get('column', '')) for ref in refs if isinstance(ref, dict)):
                        daily_change = True
                    elif not refs:
                        input_cells = [fact_cell({'value': value}, answer.get('log') or []) for value in as_numbers(claim.get('inputs'))]
                        daily_change |= bool(input_cells and all(cell and is_daily_metric(cell[2]) for cell in input_cells))
            if any(period not in daily_periods for period in requested):
                problems.append('Also return revenue per trading day for each requested month, labelled by invoice_month; total revenue alone does not answer the per-day comparison.')
            if not daily_change:
                problems.append('Include the change in revenue per trading day, using those daily values as the two inputs (not total revenue); bind the input_cells so the two comparisons stay distinct.')
    extremum = bool(re.search(r'\b(most|fewest|highest|lowest|maximum|minimum)\b', question))
    tie_sensitive = 'trading days' in question or bool(re.search(r'\b(all|ties|tied)\b|\bwhich months\b', question))
    if extremum and tie_sensitive:
        for call in calls:
            try:
                tree = sqlglot.parse_one(call['code'], read='duckdb')
                limit = tree.args.get('limit')
                if limit and isinstance(limit.expression, exp.Literal) and int(limit.expression.this) == 1:
                    problems.append('LIMIT 1 can hide tied winners for this question. Return every row equal to the requested maximum/minimum (or use DENSE_RANK = 1), including the winning measurement.')
                    break
            except Exception:
                pass
        for call in calls:
            rows = call['rows']
            day_values = [row.get('trading_days') for row in rows]
            if len(rows) > 1 and all(isinstance(value, (int, float)) for value in day_values) and len(set(day_values)) == 1:
                for index, row in enumerate(rows):
                    if not any(c.get('from_call') == call['n'] and c.get('row') == index and c.get('column') == 'trading_days' for c in claims):
                        problems.append('Include each tied month and its verified trading-day count: ' + fact_label(row, call) + '.')
    return list(dict.fromkeys(problems))


def requested_periods(question):
    """Extract explicit months, including a shared year: 'March to April 2011'."""
    import datetime
    text = str(question).lower()
    names = {datetime.date(2000, month, 1).strftime('%B').lower(): month for month in range(1, 13)}
    pattern = r'\b(' + '|'.join(names) + r')(?:\s+((?:19|20)\d{2}))?\b'
    found = list(re.finditer(pattern, text))
    years = re.findall(r'\b(?:19|20)\d{2}\b', text)
    periods = re.findall(r'\b(?:19|20)\d{2}-\d{2}\b', text)
    for index, match in enumerate(found):
        year = match[2]
        if not year:
            # A shared year belongs to the adjacent named-month phrase, not an
            # unrelated period elsewhere in the question.
            following = next((item for item in found[index + 1:] if item[2]), None)
            if following and re.fullmatch(r'[\s,]*(?:(?:to|and|with|versus|vs)[\s,]*)?', text[match.end():following.start()]):
                year = following[2]
            elif len(set(years)) == 1:
                year = years[0]
        if year:
            periods.append(f'{year}-{names[match[1]]:02d}')
    return list(dict.fromkeys(periods))

def is_daily_metric(column):
    """Recognize a stated per-day result, not a trading-day count alone."""
    return bool(re.search(r'per_?(?:trading_?)?day|daily', str(column), re.I))


def focus_verified_extrema(answer):
    """Select tied extrema from a complete result, keeping other facts as context.

    This only handles explicit trading-day extrema over a direct, unfiltered
    dim_month result. A partial or derived query cannot prove that unseen rows
    are not tied winners, so it retains the ordinary verification/repair path.
    The full result and every supported claim remain inspectable.
    """
    question = str(answer.get('question') or '').lower()
    if 'trading days' not in question or answer.get('scope_errors'):
        return answer
    largest = bool(re.search(r'\b(most|highest|maximum)\b', question))
    smallest = bool(re.search(r'\b(fewest|lowest|minimum)\b', question))
    if largest == smallest:
        return answer
    for call in answer.get('log') or []:
        if not call.get('ok') or call.get('tool') != 'run_sql' or not call.get('rows'):
            continue
        try:
            tree = sqlglot.parse_one(call.get('code', ''), read='duckdb')
            tables = list(tree.find_all(exp.Table))
            complete = (isinstance(tree, exp.Select) and len(tables) == 1 and tables[0].name.lower() == 'dim_month'
                        and len(list(tree.find_all(exp.Select))) == 1
                        and not any(tree.args.get(key) for key in ('limit', 'offset', 'where', 'having', 'qualify', 'group', 'distinct')))
            columns = {node.name for item in tree.expressions for node in [item.this if isinstance(item, exp.Alias) else item]
                       if isinstance(node, exp.Column)}
            if not complete or not {'invoice_month', 'trading_days'}.issubset(columns):
                continue
        except Exception:
            continue
        rows = call['rows']
        values = [row.get('trading_days') for row in rows]
        if not all(isinstance(value, (int, float)) and not isinstance(value, bool) and about_equal(value, value) for value in values):
            continue
        extreme = max(values) if largest else min(values)
        winners = {index for index, value in enumerate(values) if value == extreme}
        winning_claims = [claim for claim in answer.get('claims', [])
                          if claim.get('status') == 'supported' and claim.get('from_call') == call['n']
                          and claim.get('row') in winners and claim.get('column') == 'trading_days']
        if {claim.get('row') for claim in winning_claims} != winners:
            continue  # Never turn an omitted or rejected tied value into a fact.
        for claim in answer.get('claims', []):
            if claim.get('status') != 'supported':
                continue
            relevant = any(claim is winner for winner in winning_claims)
            claim['display_role'] = 'answer' if relevant else 'context'
            claim['relevance_reason'] = ('Requested tied extremum.' if relevant else
                                         'Verified context outside the requested trading-day extremum.')
        answer['answer_selection'] = {'metric': 'trading_days', 'direction': 'maximum' if largest else 'minimum',
                                      'from_call': call['n'], 'rows': sorted(winners),
                                      'basis': 'All rows from a direct, unfiltered dim_month query.'}
        break
    return answer


def verified_measurement_unit(column):
    """Only expose units justified by a known measurement column."""
    column = str(column).lower()
    if any(word in column for word in ('pct', 'percent', 'cancellation_rate')):
        return '%'
    money = ('revenue' in column and not any(word in column for word in ('share', 'change', 'growth')))
    if money or column in ('aov', 'average_order_value'):
        return 'GBP per trading day' if is_daily_metric(column) else 'GBP'
    return {'trading_days': 'days', 'orders': 'orders', 'customers': 'customers',
            'units': 'units', 'quantity': 'units'}.get(column, '')


def country_scope_problem(sql, question, countries=None):
    """Check explicitly named countries against SQL populations, never answers.

    The vocabulary comes from this database, so the rule applies to any market.
    Country shares may have an unrestricted denominator; their numerator must
    still select the requested country. This is a bounded intent guard.
    """
    if countries is None:
        try:
            with duckdb.connect(DB, read_only=True, config={'enable_external_access': 'false'}) as con:
                countries = [row[0] for row in con.execute('SELECT DISTINCT country FROM sales').fetchall()]
        except Exception:
            return None  # Database availability is reported by run_sql itself.
    vocabulary = {str(country).casefold(): str(country) for country in countries if country is not None}
    aliases = {'uk': 'United Kingdom', 'u.k.': 'United Kingdom'}
    names = dict(vocabulary)
    names.update({alias: country for alias, country in aliases.items() if country.casefold() in vocabulary})
    text = str(question or '').casefold()
    included, excluded = set(), set()
    country_names = '|'.join(re.escape(name) for name in sorted(names, key=len, reverse=True))
    for name, country in names.items():
        for mention in re.finditer(r'(?<!\w)' + re.escape(name) + r'(?!\w)', text):
            prefix = text[max(0, mention.start()-180):mention.start()]
            negative = bool(re.search(r'(?:outside(?:\s+of)?|excluding|except(?:\s+for)?|other\s+than|not\s+(?:from|in))\s+(?:the\s+)?(?:(?:' + country_names + r')\s*(?:,|and|or)\s*)*$', prefix))
            (excluded if negative else included).add(country)
    if not included and not excluded:
        return None
    universe = set(vocabulary.values())
    tree = sqlglot.parse_one(sql, read='duckdb')

    def domain(predicate):
        if isinstance(predicate, (exp.Where, exp.Having, exp.Paren)):
            return domain(predicate.this)
        if isinstance(predicate, exp.And):
            return domain(predicate.this) & domain(predicate.expression)
        if isinstance(predicate, exp.Or):
            return domain(predicate.this) | domain(predicate.expression)
        if isinstance(predicate, exp.Not):
            columns = list(predicate.this.find_all(exp.Column))
            if columns and all(column.name.lower() == 'country' for column in columns):
                return universe - domain(predicate.this)
            return universe
        if isinstance(predicate, (exp.EQ, exp.NEQ)):
            left, right = predicate.this, predicate.expression
            if isinstance(right, exp.Column):
                left, right = right, left
            if isinstance(left, exp.Column) and left.name.lower() == 'country' and isinstance(right, exp.Literal) and right.is_string:
                chosen = {vocabulary.get(str(right.this).casefold(), str(right.this))}
                return universe - chosen if isinstance(predicate, exp.NEQ) else chosen
        if isinstance(predicate, exp.In) and isinstance(predicate.this, exp.Column) and predicate.this.name.lower() == 'country' and predicate.expressions:
            if all(isinstance(value, exp.Literal) and value.is_string for value in predicate.expressions):
                return {vocabulary.get(str(value.this).casefold(), str(value.this)) for value in predicate.expressions}
        return universe

    restrictions = [domain(predicate) for predicate in tree.find_all(exp.Where, exp.Having)]
    restrictions = [values for values in restrictions if values != universe]
    country_cases = [case for case in tree.find_all(exp.If)
                     if any(column.name.lower() == 'country' for column in case.this.find_all(exp.Column))]
    case_restrictions = [domain(case.this) for case in country_cases]
    candidates = restrictions or case_restrictions
    share_question = bool(re.search(r'\bshare\b|\bproportion\b|\b(?:percentage|percent)\b.*\b(?:from|of)\b', text))
    country_filtered = any(column.name.lower() == 'country' for predicate in tree.find_all(exp.Where, exp.Having) for column in predicate.find_all(exp.Column))
    if share_question and not candidates and not country_filtered:
        return None  # A separate query may fetch the unrestricted denominator.
    allowed = (included or universe) - excluded
    expected = ', '.join(sorted(included)) if included else 'all countries except ' + ', '.join(sorted(excluded))
    if not candidates or any(not values or not values <= allowed for values in candidates):
        return ('The question requests ' + expected + ', but the SQL country population differs or is missing. '
                'Use the requested full country names in WHERE/IN; comparisons may query each requested country separately. '
                'For a share, select the requested country in CASE WHEN for the numerator and keep the all-country denominator.')
    # A WHERE country restriction scopes every measurement. Without one, each
    # copied measure needs a country-specific CASE; only shares need a whole.
    if not restrictions and not share_question:
        for aggregate in tree.find_all(exp.AggFunc):
            for column in aggregate.find_all(exp.Column):
                if column.name.lower() not in {'revenue', 'quantity', 'invoice_no', 'customer_id'}:
                    continue
                current, scoped = column, False
                while current is not aggregate and current.parent is not None:
                    parent = current.parent
                    if isinstance(parent, exp.If) and parent.args.get('true') is current and domain(parent.this) <= allowed:
                        scoped = True
                        break
                    current = parent
                if not scoped:
                    return ('A requested-country measure still includes other countries. Restrict it with WHERE country/IN, '
                            'or CASE WHEN country matches THEN measure ELSE 0/NULL END. Keep all-country totals only for a share denominator.')
    return None


def public_prose_texts(answer):
    """Every displayed prose field shares the same numeric exposure checks."""
    texts = [str(answer.get(key) or '') for key in
             ('findings', 'limitations', 'interpretation')]
    steps = answer.get('suggested_next_steps') or []
    # Treat an unexpected scalar as visible text too; malformed types must not
    # silently escape checking when a UI chooses to display their string form.
    if not isinstance(steps, (list, tuple)):
        steps = [steps]
    return texts + [str(step) for step in steps]



def narrative_facts(answer):
    """Read facts from exact verified cells, never from the model's prose."""
    facts = []
    for claim in claims_to_show(answer):
        if claim.get('calc') not in (None, 'none'):
            continue
        cell = fact_cell(claim, answer.get('log') or [])
        if cell:
            call, index, column, row = cell
            facts.append({'claim': claim, 'call': call, 'row': row, 'column': column,
                          'value': claim['value'], 'label': fact_label(row, call)})
    return facts

def narrative_measure(column, call=None):
    """Give familiar names to the measurements used in a sentence."""
    column = str(column).lower()
    invoice_measure = invoice_count_measure(column, call)
    if invoice_measure:
        return invoice_measure
    if 'revenue' in column and not any(part in column for part in ('share', 'percent', 'pct', 'change')):
        return 'revenue per trading day' if is_daily_metric(column) else 'revenue'
    return {'aov': 'average order value', 'average_order_value': 'average order value',
            'customers': 'identified customers',
            'units': 'units sold', 'quantity': 'units sold', 'trading_days': 'trading days'}.get(column, '')

def narrative_scope(answer):
    """Keep follow-up questions self-contained using the checked year scope."""
    years = list(dict.fromkeys(re.findall(r'\b(?:19|20)\d{2}\b', str(answer.get('question') or ''))))
    sql = ' '.join(str(call.get('code', '')) for call in answer.get('log', [])
                   if call.get('ok') and call.get('tool') == 'run_sql')
    # A suggested year is context, not a new result. Require it in both the
    # request and its successful SQL rather than inventing a future target.
    if len(years) == 1 and years[0] in sql:
        return 'in ' + years[0]
    return 'across the available dataset'

def narrative_country(fact):
    """Keep a known country filter in the next question, including scalar SQL."""
    if fact['row'].get('country'):
        return str(fact['row']['country'])
    try:
        tree = sqlglot.parse_one(fact['call'].get('code') or '', read='duckdb')
        countries = {node.expression.this for where in tree.find_all(exp.Where) for node in where.find_all(exp.EQ)
                     if isinstance(node.this, exp.Column) and node.this.name == 'country'
                     and isinstance(node.expression, exp.Literal) and node.expression.is_string}
        if len(countries) == 1:
            return next(iter(countries))
    except Exception:
        pass
    return ''

def narrative_calculation(claim, answer):
    """Explain a checked comparison without inventing its cause or magnitude."""
    operation = claim.get('calc') or 'none'
    if operation not in ('pct_change', 'difference', 'diff'):
        return None
    inputs = as_numbers(claim.get('inputs'))
    if len(inputs) != 2:
        return None
    cells = []
    for index, value in enumerate(inputs):
        refs = claim.get('input_cells') or []
        ref = dict(refs[index], value=value) if index < len(refs) and isinstance(refs[index], dict) else {'value': value}
        cell = fact_cell(ref, answer.get('log') or [])
        if not cell:
            return None
        cells.append(cell)
    labels = [fact_label(cell[3], cell[0]) for cell in cells]
    measures = [narrative_measure(cell[2], cell[0]) for cell in cells]
    if not all(labels) or not measures[0] or measures[0] != measures[1]:
        return None
    measure = measures[0]
    value = claim['value']
    direction = 'higher' if value > 0 else 'lower' if value < 0 else 'unchanged'
    amount = fact_value(value, unit='%' if operation == 'pct_change' else
                        'GBP' if str(claim.get('unit') or '').startswith('GBP') else '')
    # Keep a negative result signed, matching the exact verified claim. Writing
    # an absolute decline would otherwise introduce a new displayed number.
    if direction == 'unchanged':
        sentence = f'{measure.capitalize()} was unchanged between {labels[0]} and {labels[1]} ({amount}).'
    else:
        noun = 'change' if operation == 'pct_change' else 'difference'
        sentence = f'{measure.capitalize()} was {direction} in {labels[0]} than in {labels[1]}, a {noun} of {amount}.'
    old_text = str(claim.get('text') or '')
    if old_text.startswith(('Yes.', 'No.')):
        sentence = old_text.split('.', 1)[0] + '. ' + sentence
    return {'sentence': sentence, 'cells': cells, 'measure': measure,
            'periods': [str(cell[3].get('invoice_month') or '') for cell in cells]}

def public_completeness_note(answer):
    """Describe omissions to a reader; keep repair instructions in the audit."""
    issues = ' '.join(str(item) for item in answer.get('completeness_issues') or []).lower()
    if not issues:
        return ''
    missing = []
    if 'per trading day' in issues or 'per-day' in issues:
        missing.append('the comparison per trading day')
    if 'percentage change' in issues:
        missing.append('the percentage change')
    if 'difference in pounds' in issues:
        missing.append('the revenue difference')
    if 'share' in issues:
        missing.append('the requested share')
    if 'tied' in issues or 'tie' in issues:
        missing.append('whether other results share the lead')
    if 'ranked' in issues or 'result row' in issues:
        missing.append('the full requested ranking')
    if 'each requested month' in issues:
        missing.append('figures for every requested month')
    if missing:
        return 'This is a partial answer: ' + ', '.join(dict.fromkeys(missing)) + ' still needs verification.'
    return 'Some requested details still need verification. The figures shown are a partial answer.'

def apply_human_narrative(answer):
    """Add concise meaning and next steps while preserving all verified claims.

    These small rules describe the evidence already returned. They do not run
    queries, reuse model wording, infer causes or change the numerical answer.
    """
    facts = narrative_facts(answer)
    visible = claims_to_show(answer)
    scope = narrative_scope(answer)
    answer['interpretation'] = ''
    answer['suggested_next_steps'] = []
    gap = data_gap_reason(answer.get('question', ''))
    if not visible:
        if gap:
            question = str(answer.get('question') or '').lower()
            if re.search(r'profit|margin|cost|cogs', question):
                answer['interpretation'] = 'Sales totals are available, but calculating profitability also requires cost data.'
                answer['suggested_next_steps'] = ['What was total revenue across the available dataset?',
                                                  'Add validated cost data before calculating profit or margin.']
            elif re.search(r'forecast|predict|next (?:month|quarter|year)', question):
                answer['interpretation'] = 'The available records can describe past sales. This analysis does not provide a verified forecast.'
                answer['suggested_next_steps'] = ['How did revenue change by month across the available dataset?',
                                                  'Review historical trends before building and testing a separate forecasting model.']
            elif 'incomplete' in gap.lower():
                answer['interpretation'] = 'The reporting periods are not directly comparable as full months. A daily view can make the available activity easier to compare.'
                periods = requested_periods(answer.get('question', ''))
                labels = [fact_label({'invoice_month': period}) for period in periods]
                followup = ('What was revenue per trading day in ' + ' and '.join(labels) + '?') if labels else 'Which reporting months are complete in the dataset?'
                answer['suggested_next_steps'] = [followup, 'Check the reporting coverage before comparing monthly totals.']
            else:
                answer['interpretation'] = 'The requested conclusion depends on information outside these sales records.'
                answer['suggested_next_steps'] = ['Which countries generated the most revenue across the available dataset?',
                                                  'Identify the missing data before extending the analysis.']
        elif 'No usable data' in str(answer.get('findings')):
            answer['interpretation'] = 'No matching measurements were returned. This does not establish that sales were zero.'
            answer['suggested_next_steps'] = ['Which reporting months are available in the dataset?',
                                              'Check the requested dates and names against the data coverage.']
        else:
            answer['interpretation'] = 'The available evidence was not sufficient to answer reliably. A narrower question may return a usable result.'
            answer['suggested_next_steps'] = ['What was total revenue across the available dataset?',
                                              'Ask about a single measure and a specific reporting period.']
        return answer

    completeness = [fact for fact in facts if fact['column'] == 'is_complete_month' and isinstance(fact['value'], bool)]
    if completeness and re.search(r'complete|partial|incomplete', str(answer.get('question') or ''), re.I):
        fact = completeness[0]
        label = fact['label'] or 'The requested period'
        answer['findings'] = ('Yes, ' + label + ' is a complete reporting month.' if fact['value'] else
                              'No, ' + label + ' is an incomplete reporting month.')
        days = next((item for item in facts if item['column'] == 'trading_days' and item['label'] == fact['label']), None)
        if days:
            answer['findings'] += ' It contains ' + fact_value(days['value']) + ' trading days.'
        answer['interpretation'] = ('Full-month totals can be compared, while allowing for differences in trading-day counts.' if fact['value'] else
                                     'A partial month understates the coverage of a full reporting period. Use revenue per trading day when comparing the available activity.')
        answer['suggested_next_steps'] = ['What was revenue per trading day in ' + label + '?',
                                          'Check trading-day counts before interpreting a monthly change.']
        return answer

    comparisons = [item for claim in visible if (item := narrative_calculation(claim, answer))]
    if comparisons:
        sentences = list(dict.fromkeys(item['sentence'] for item in comparisons))
        answer['findings'] = ' '.join(sentences[:2])
        compared_months = {period for item in comparisons for period in item['periods'] if period}
        days = [fact for fact in facts if fact['column'] == 'trading_days'
                and str(fact['row'].get('invoice_month') or '') in compared_months]
        partial = any(fact['value'] is False for fact in facts if fact['column'] == 'is_complete_month')
        if partial:
            answer['interpretation'] = 'A compared month is incomplete. Treat full-month totals cautiously and use revenue per trading day to compare the available activity.'
        elif len({fact['row'].get('invoice_month') for fact in days}) >= 2:
            if len({fact['value'] for fact in days}) > 1:
                answer['interpretation'] = 'The months have different numbers of trading days. Compare the daily figures before treating a change in total revenue as a change in daily sales activity.'
            else:
                answer['interpretation'] = 'The months have the same number of trading days. Their reported difference is not explained by a difference in reporting length; the figures alone do not establish its cause.'
        elif compared_months:
            answer['interpretation'] = 'The direction of change is supported by the returned figures. Trading-day counts would help distinguish reporting length from daily sales activity.'
        else:
            answer['interpretation'] = 'This describes the difference in the recorded measure between the selected groups. A product breakdown can show which items contributed to the gap.'
        if compared_months:
            labels = [fact_label({'invoice_month': period}) for period in sorted(compared_months)]
            answer['suggested_next_steps'] = ['What was revenue per trading day in ' + ' and '.join(labels) + '?',
                                              'Which products contributed the most revenue ' + scope + '?']
        else:
            country = next((cell[3].get('country') for item in comparisons for cell in item['cells'] if cell[3].get('country')), None)
            where = (' in ' + str(country)) if country else ''
            answer['suggested_next_steps'] = ['Which products contributed the most revenue' + where + ' ' + scope + '?',
                                              'Compare order counts and average order value before interpreting the revenue gap.']
        return answer

    shares = [claim for claim in visible if claim.get('calc') == 'share' or
              (re.search(r'share|proportion', str(claim.get('column') or ''), re.I) and claim.get('unit') == '%')]
    if shares:
        claim = shares[0]
        country = narrative_share_country(claim, answer)
        answer['findings'] = (country or 'The selected group') + ' accounted for ' + fact_value(claim['value'], unit='%') + ' of the reported total.'
        if 50 < claim['value'] <= 100:
            answer['interpretation'] = 'Most of the reported total is concentrated in this group. Its performance therefore has a large influence on the overall result.'
        elif 0 <= claim['value'] < 50:
            answer['interpretation'] = 'The remaining groups contribute more than this group. Compare the other groups to understand how the total is distributed.'
        elif claim['value'] == 50:
            answer['interpretation'] = 'This group contributes the same share as the rest combined.'
        else:
            answer['interpretation'] = 'Review the numerator and denominator before treating this result as a conventional share of a positive total.'
        answer['suggested_next_steps'] = ['Which countries contributed the most revenue ' + scope + '?',
                                          'Compare shares over time before describing concentration as a lasting trend.']
        return answer

    ranked = [fact for fact in facts if narrative_measure(fact['column'], fact['call']) and
              any(fact['row'].get(key) is not None for key in ('country', 'description', 'stock_code', 'customer_id'))]
    rank_request = bool(re.search(r'\btop\b|\brank|\bmost\b|\bhighest\b|\bleading\b', str(answer.get('question') or ''), re.I))
    if ranked and rank_request:
        groups = {}
        for fact in ranked:
            groups.setdefault((fact['call']['n'], fact['column']), []).append(fact)
        candidates = max(groups.values(), key=len)
        high = max(fact['value'] for fact in candidates)
        leaders = list({fact['label']: fact for fact in candidates if about_equal(fact['value'], high)}.values())
        first = leaders[0]
        measure = narrative_measure(first['column'], first['call'])
        names = ' and '.join(fact['label'] for fact in leaders[:3]) if len(leaders) <= 3 else 'Several returned items'
        amount = fact_value(high, first['column'])
        answer['findings'] = names + (' share the lead' if len(leaders) > 1 else ' leads')
        entity = 'customers' if first['row'].get('customer_id') is not None else 'markets' if first['row'].get('country') is not None else 'products'
        answer['findings'] += ' among the ' + entity + ' shown, with ' + amount + (' in ' if 'revenue' in measure else ' ') + measure + '.'
        if len(candidates) > len(leaders):
            answer['findings'] += ' The full ranking is available in the detailed figures.'
        if first['row'].get('customer_id') is not None:
            answer['interpretation'] = 'Use this ranking to prioritize account review. Order counts and revenue describe different aspects of customer activity; profitability also requires cost data.'
            answer['suggested_next_steps'] = ['Which customers generated the most revenue ' + scope + '?',
                                              'Review customer order frequency alongside order value.']
        elif first['row'].get('country') is not None:
            answer['interpretation'] = 'This identifies markets worth examining in more detail. Product and order-value breakdowns can explain how the recorded totals are composed.'
            answer['suggested_next_steps'] = ['Which products generated the most revenue in ' + str(first['row']['country']) + ' ' + scope + '?',
                                              'Compare market revenue with completed-order counts.']
        else:
            answer['interpretation'] = 'This identifies the strongest recorded products for the selected measure. Use it as a starting point for stock review alongside cancellation values and product costs.'
            country = narrative_country(first)
            where = (' in ' + country) if country else ''
            answer['suggested_next_steps'] = ['Which products sold the most units' + where + ' ' + scope + '?',
                                              'Review cancellation values before changing stock priorities.']
        return answer

    primary_facts = [fact for fact in facts if narrative_measure(fact['column'], fact['call']) and fact['column'] != 'trading_days']
    unique_facts = {(fact['call']['n'], fact['claim'].get('row'), fact['column']): fact for fact in primary_facts}
    if len(unique_facts) > 1:
        measures = {narrative_measure(fact['column'], fact['call']) for fact in unique_facts.values()}
        measure = next(iter(measures)) if len(measures) == 1 else 'the requested measurements'
        if all(fact['row'].get('invoice_month') for fact in unique_facts.values()):
            entity = 'reporting periods'
        elif all(fact['row'].get('country') for fact in unique_facts.values()):
            entity = 'markets'
        elif all(fact['row'].get('customer_id') is not None for fact in unique_facts.values()):
            entity = 'customers'
        else:
            entity = 'groups'
        answer['findings'] = 'The verified breakdown of ' + measure + ' across the ' + entity + ' is available in the detailed figures.'
        answer['interpretation'] = ('The individual figures are verified, but the requested comparison is not yet complete.'
                                    if answer.get('completeness_issues') else
                                    'Use the breakdown to compare the recorded groups or periods. A separate checked calculation is needed to quantify a change.')
        answer['suggested_next_steps'] = ['Which products contributed the most revenue ' + scope + '?',
                                          'Choose the groups or periods to compare and ask for their difference or percentage change.']
        return answer
    primary = next(iter(unique_facts.values()), None)
    if primary:
        value, column, label = primary['value'], primary['column'], primary['label']
        amount = fact_value(value, column)
        measure = narrative_measure(column, primary['call'])
        if measure == 'revenue':
            answer['findings'] = (label + ' generated ' if label else 'Revenue was ') + amount + (' in revenue.' if label else '.')
            answer['interpretation'] = 'This is the sales baseline for the requested scope. A monthly breakdown shows whether activity is steady or concentrated in particular periods.'
            country = narrative_country(primary)
            where = (' in ' + str(country)) if country else ''
            answer['suggested_next_steps'] = ['How did revenue change by month' + where + ' ' + scope + '?',
                                              'Which products contributed the most revenue' + where + ' ' + scope + '?']
        elif measure == 'completed orders':
            answer['findings'] = (label + ' placed ' if label else 'There were ') + amount + ' completed orders.'
            answer['interpretation'] = 'Each completed invoice counts as an order, even when it contains several items. Read order counts alongside average order value to understand sales activity.'
            who = (' for ' + label) if label.startswith(('Customer ', 'Stock code ')) else ''
            answer['suggested_next_steps'] = ['What was average order value' + who + ' ' + scope + '?',
                                              'How did completed orders change by month ' + scope + '?']
        elif measure == 'cancellation invoices':
            answer['findings'] = (label + ' had ' if label else 'There were ') + amount + ' cancellation invoices.'
            answer['interpretation'] = 'Each distinct cancellation invoice is counted once, even when it contains several lines. These records do not identify which original orders were later cancelled.'
            answer['suggested_next_steps'] = ['What was the value of cancelled sales ' + scope + '?',
                                              'Review cancellation values alongside invoice counts.']
        elif measure in ('invoices', 'count'):
            answer['findings'] = ('There were ' + amount + ' distinct invoices.' if measure == 'invoices' else
                                  'The returned count was ' + amount + '.')
            answer['interpretation'] = ('This counts distinct invoice numbers. The query does not establish a completed-only or cancellation-only population.' if measure == 'invoices' else
                                         'This is the count returned by the query. Its evidence does not establish a count of distinct completed or cancellation invoices.')
            answer['suggested_next_steps'] = ['How many orders did we take ' + scope + '?',
                                              'Specify whether the count should include completed orders or cancellation invoices.']
        elif measure == 'identified customers':
            answer['findings'] = amount + ' identified customers made purchases.'
            answer['interpretation'] = 'This counts buyers with a recorded customer ID. Transactions without an ID cannot establish additional distinct customers.'
            answer['suggested_next_steps'] = ['Which customers placed the most orders ' + scope + '?',
                                              'Review how often identified customers returned to purchase.']
        elif measure == 'units sold':
            answer['findings'] = 'Sales totalled ' + amount + ' units.'
            answer['interpretation'] = 'Unit volume describes sales activity independently of item prices. Compare it with revenue to see whether value and volume move together.'
            answer['suggested_next_steps'] = ['Which products sold the most units ' + scope + '?',
                                              'How did revenue change by month ' + scope + '?']
        elif measure == 'average order value':
            answer['findings'] = 'The average completed order was worth ' + amount + '.'
            answer['interpretation'] = 'Average order value helps separate order size from order volume. Read it alongside completed-order counts when assessing sales changes.'
            answer['suggested_next_steps'] = ['How many completed orders were recorded ' + scope + '?',
                                              'How did average order value change by month ' + scope + '?']
    elif any(fact['column'] == 'cancellation_rate' for fact in facts):
        fact = next(fact for fact in facts if fact['column'] == 'cancellation_rate')
        answer['findings'] = 'Cancellation invoices were ' + fact_value(fact['value'], unit='%') + ' of completed orders.'
        answer['interpretation'] = 'This compares invoice counts in the same period. It cannot show which original orders were later cancelled, because those links are not available.'
        answer['suggested_next_steps'] = ['What was the value of cancelled sales ' + scope + '?',
                                          'Review cancellation values alongside invoice counts.']

    if not answer['interpretation']:
        if answer.get('answer_selection', {}).get('metric') == 'trading_days':
            answer['interpretation'] = 'These results identify the requested trading-day extreme. Trading-day counts help put monthly sales totals on a comparable basis.'
            answer['suggested_next_steps'] = ['How did revenue per trading day vary by month across the available dataset?',
                                              'Check reporting coverage before comparing monthly sales totals.']
        else:
            answer['interpretation'] = 'The evidence supports these measurements. It does not establish what caused them.'
            answer['suggested_next_steps'] = ['How did revenue change by month ' + scope + '?',
                                              'Review the detailed figures before drawing a broader conclusion.']
    return answer


def narrative_share_country(claim, answer):
    """Name an unambiguous country from the bound share or numerator cell."""
    log = answer.get('log') or []
    if claim.get('calc') == 'share':
        inputs = as_numbers(claim.get('inputs'))
        if not inputs:
            return ''
        refs = claim.get('input_cells') or []
        reference = dict(refs[0], value=inputs[0]) if refs and isinstance(refs[0], dict) else {'value': inputs[0]}
    else:
        reference = claim
    cell = fact_cell(reference, log)
    if not cell:
        return ''
    call, index, column, row = cell
    country = narrative_country({'call': call, 'row': row})
    if country:
        return country
    # A wide aggregate often keeps the country in SUM(CASE ...) instead of a
    # country column. Only inspect this bound projection and positive branches
    # with a zero/null ELSE; an exclusion such as CASE ... THEN 0 ELSE revenue
    # must not be named as that country's contribution.
    try:
        tree = sqlglot.parse_one(call.get('code') or '', read='duckdb')
        projections = [item for item in tree.expressions if item.alias_or_name == column]
        countries = set()
        for projection in projections:
            for case in projection.find_all(exp.Case):
                default = case.args.get('default')
                if default is not None and not isinstance(default, exp.Null) and not (
                        isinstance(default, exp.Literal) and not default.is_string and float(default.this) == 0):
                    continue
                for branch in case.args.get('ifs') or []:
                    contribution = branch.args.get('true')
                    if isinstance(contribution, exp.Null) or (isinstance(contribution, exp.Literal)
                            and not contribution.is_string and float(contribution.this) == 0):
                        continue
                    for condition in branch.this.find_all(exp.EQ):
                        if (isinstance(condition.this, exp.Column) and condition.this.name == 'country'
                                and isinstance(condition.expression, exp.Literal) and condition.expression.is_string
                                and condition.find_ancestor(exp.Not) is None):
                            countries.add(condition.expression.this)
        return next(iter(countries)) if len(countries) == 1 else ''
    except Exception:
        return ''


def invoice_count_measure(column, call=None):
    """Name an invoice count from its query population, never from its alias alone.

    Only a direct distinct-invoice count with a clear Boolean WHERE condition
    establishes completed or cancellation invoices. Complex shapes stay neutral.
    """
    column = str(column).lower()
    if not re.fullmatch(r'(?:orders?|invoices?|count|(?:order|invoice|cancellation|cancelled|completed)_(?:orders?|invoices?|count))', column):
        return ''
    try:
        tree = sqlglot.parse_one((call or {}).get('code') or '', read='duckdb')
        if not isinstance(tree, exp.Select):
            return 'count'
        matches = [item for item in tree.expressions if item.alias_or_name.lower() == column]
        expression = matches[0].this if len(matches) == 1 and isinstance(matches[0], exp.Alias) else None
        if not isinstance(expression, exp.Count) or not isinstance(expression.this, exp.Distinct):
            return 'count'
        values = expression.this.expressions
        if len(values) != 1 or not isinstance(values[0], exp.Column) or values[0].name not in ('invoice_no', 'invoice'):
            return 'count'

        def population(node):
            if isinstance(node, (exp.Where, exp.Paren)):
                return population(node.this)
            if isinstance(node, exp.Column) and node.name == 'is_cancellation':
                return {True}
            if isinstance(node, exp.Not):
                return {not value for value in population(node.this)}
            if isinstance(node, exp.And):
                return population(node.this) & population(node.expression)
            if isinstance(node, exp.Or):
                return population(node.this) | population(node.expression)
            if isinstance(node, (exp.EQ, exp.Is, exp.NEQ)):
                left, right = node.this, node.expression
                if isinstance(right, exp.Column):
                    left, right = right, left
                if isinstance(left, exp.Column) and left.name == 'is_cancellation' and isinstance(right, exp.Boolean):
                    value = bool(right.this)
                    return {not value if isinstance(node, exp.NEQ) else value}
            return {True, False}

        selected = population(tree.args.get('where'))
        if selected == {True}:
            return 'cancellation invoices'
        if selected == {False}:
            return 'completed orders'
        return 'invoices'
    except Exception:
        return 'count'



In [2]:
import yaml
import pandas as pd
import matplotlib.pyplot as plt

questions = yaml.safe_load(open('../eval/benchmark.yaml'))['questions']
print(len(questions), 'questions')


30 questions


## 1. Run everything


In [ ]:
from copy import deepcopy
from pathlib import Path

def evaluation_same(a, b):
    """Use one tolerance for facts; booleans and identifiers are not measurements."""
    import math
    if isinstance(a, bool) or isinstance(b, bool):
        return type(a) is bool and type(b) is bool and a is b
    try:
        return math.isclose(float(a), float(b), rel_tol=1e-6, abs_tol=.011)
    except (TypeError, ValueError, OverflowError):
        return False

def evaluation_labels(key, value):
    """Canonical labels accept SQL dates, month names and integer-looking IDs."""
    import calendar
    text = str(value).strip().lower()
    labels = [text]
    if key in ('customer_id', 'stock_code', 'invoice_no'):
        try:
            if float(value).is_integer():
                labels.append(str(int(float(value))))
        except (ValueError, TypeError, OverflowError):
            pass
    if key == 'invoice_month' and re.fullmatch(r'\d{4}-\d{2}', text):
        year, month = map(int, text.split('-'))
        if 1 <= month <= 12:
            labels += [f'{calendar.month_name[month]} {year}'.lower(),
                       f'{calendar.month_abbr[month]} {year}'.lower()]
    return labels

def evaluation_has_label(text, labels):
    return any(re.search(r'(?<!\w)' + re.escape(label) + r'(?!\w)', str(text).lower())
               for label in labels)

def evaluation_bound_claims(claims, log):
    """Bind identical displayed values symmetrically; never filter lower-tier claims.

    This is evaluation-only normalization, not verification or answer repair.
    A unique SQL cell can supply a missing row/column, independent of claim status.
    Explicit invalid coordinates are preserved so they cannot be silently repaired.
    """
    from copy import deepcopy
    result = deepcopy(claims or [])
    for claim in result:
        if claim.get('row') is not None or claim.get('column') is not None:
            continue
        cells = []
        for call in log or []:
            if call.get('tool') != 'run_sql' or not call.get('ok'):
                continue
            if claim.get('from_call') is not None and claim['from_call'] != call.get('n'):
                continue
            for index, row in enumerate(call.get('rows') or []):
                for column, value in row.items():
                    if column not in ('customer_id', 'stock_code', 'invoice_no') and evaluation_same(claim.get('value'), value):
                        cells.append((call['n'], index, column))
        if len(cells) == 1:
            claim.update(zip(('from_call', 'row', 'column'), cells[0]))
    return result

def evaluation_measurements(rows):
    """Enumerate typed ground-truth measurements and their row/column labels."""
    from decimal import Decimal
    identifiers = {'customer_id', 'stock_code', 'invoice_no'}
    standard = {'revenue', 'net_revenue', 'gross_revenue', 'orders', 'units', 'customers',
                'aov', 'trading_days', 'is_complete_month', 'pct', 'pct_change', 'uk_pct',
                'total_pct', 'per_day_pct', 'difference', 'value', 'share', 'ratio', 'count'}
    facts = []
    for row in rows or []:
        for key, value in row.items():
            if key in identifiers or not isinstance(value, (int, float, bool, Decimal)):
                continue
            labels = []
            for label_key in ('stock_code', 'description', 'invoice_month', 'country', 'customer_id'):
                if label_key not in row or label_key == 'description' and 'stock_code' in row:
                    continue
                choices = evaluation_labels(label_key, row[label_key])
                if label_key == 'stock_code' and row.get('description'):
                    choices += evaluation_labels('description', row['description'])
                labels.append((label_key, choices))
            # Wide country comparisons encode the entity in the column alias.
            if key not in standard and evaluation_metric(key) is None:
                labels.append(('column', [key.replace('_', ' ').lower()]))
            facts.append({'row': row, 'column': key, 'value': value, 'labels': labels})
    return facts

def evaluation_claim_matches(claim, fact, all_facts, log):
    if not evaluation_same(claim.get('value'), fact['value']):
        return False
    if evaluation_metric_problem(claim, fact):
        return False
    if isinstance(fact['value'], bool) and claim.get('kind') != 'boolean':
        return False
    text = str(claim.get('text') or '')
    found, _ = claim_cell(claim, log or [])
    source = next((call for call in log or [] if call.get('n') == claim.get('from_call') and call.get('ok')), None)
    bound_row = (source['rows'][claim['row']] if found else {})
    for label_key, choices in fact['labels']:
        # A source pointer cannot excuse an explicitly contradictory label.
        competing = [labels for other in all_facts for kind, labels in other['labels']
                     if kind == label_key and not set(labels).intersection(choices)]
        if any(evaluation_has_label(text, labels) for labels in competing):
            return False
        if evaluation_has_label(text, choices):
            continue
        if label_key == 'column':
            if not found or claim.get('column') != fact['column']:
                return False
        elif label_key not in bound_row or not set(evaluation_labels(label_key, bound_row[label_key])).intersection(choices):
            return False
    return True

def result_matches(claims, expected_rows, log=None):
    """Complete, label-aware row matching with identical binding for every tier."""
    facts = evaluation_measurements(expected_rows)
    bound = evaluation_bound_claims(claims, log)
    return bool(facts) and all(any(evaluation_claim_matches(c, fact, facts, log)
                                  for c in bound) for fact in facts)

def evaluation_label_claim(claim, rows):
    """Older model outputs sometimes encode a month as 201102; it is a label."""
    if str(claim.get('unit') or '').lower() not in ('month', 'date', 'invoice_month', 'customer_id', 'stock_code'):
        return False
    for row in rows:
        for key in ('invoice_month', 'customer_id', 'stock_code'):
            if key not in row:
                continue
            if not evaluation_has_label(claim.get('text', ''), evaluation_labels(key, row[key])):
                continue
            value = str(row[key]).replace('-', '')
            if str(claim.get('value')) == value or evaluation_same(claim.get('value'), value):
                return True
    return False

def score_answer(answer, question, expected_rows=None):
    """Evaluate public facts separately from provenance and SQL quality.

    Optional reference context is checked when stated, but need not be repeated.
    Unknown extra measurements fail strict factual accuracy and are labelled
    unassessed rather than falsely claiming that missing evidence proves a lie.
    No expected values or reference SQL enter the inference functions.
    """
    from copy import deepcopy
    expected = evaluation_expectations(question) if expected_rows is None else expected_rows
    public = deepcopy(answer)
    if answer.get('tier') == 3:
        public['claims'] = claims_to_show(answer)
    # Match dashboard visibility: trace annotations are metadata, not business facts.
    for key in ('findings', 'limitations', 'interpretation'):
        text = re.sub(r'\[call\s+(?:\d+|\[unverified\])\]', '', str(public.get(key) or ''), flags=re.I)
        public[key] = re.sub(r'\b(?:from_call\s*:|from[_ ]run_sql\b).*', '', text, flags=re.I | re.S).strip()
    steps = public.get('suggested_next_steps') or []
    if not isinstance(steps, (list, tuple)):
        steps = [steps]
    public['suggested_next_steps'] = []
    for step in steps:
        text = re.sub(r'\[call\s+(?:\d+|\[unverified\])\]', '', str(step), flags=re.I)
        public['suggested_next_steps'].append(re.sub(r'\b(?:from_call\s*:|from[_ ]run_sql\b).*', '', text, flags=re.I | re.S).strip())
    bound = evaluation_bound_claims(public.get('claims', []), public.get('log', []))
    required = question.get('required_columns')
    optional = set(question.get('optional_columns') or [])
    projected = [{key: value for key, value in row.items()
                  if key not in optional and (required is None or key in required
                     or key in ('stock_code', 'description', 'invoice_month', 'country', 'customer_id'))}
                 for row in expected]
    required_ok = result_matches(bound, projected, public.get('log', []))
    # Alternatives are explicit complete row sets, not a waiver for a missing list.
    required_ok = required_ok or any(result_matches(bound, rows, public.get('log', []))
                                    for rows in question.get('alternative_rows', []))
    facts = evaluation_measurements(expected + [row for rows in question.get('alternative_rows', []) for row in rows])
    contradicted, unassessed = [], []
    for claim in bound:
        if evaluation_label_claim(claim, expected):
            continue
        if re.search(r'\b(profit|margin|cost)\b', str(claim.get('text') or ''), re.I) and not any(re.search(r'profit|margin|cost', f['column'], re.I) for f in facts):
            contradicted.append(claim.get('text'))
            continue
        matches = [f for f in facts if evaluation_same(claim.get('value'), f['value'])]
        if matches:
            if not any(evaluation_claim_matches(claim, f, facts, public.get('log', [])) for f in matches):
                contradicted.append(claim.get('text') or str(claim.get('value')))
            continue
        found, value = claim_cell(claim, public.get('log', []))
        if found and evaluation_same(claim.get('value'), value):
            if evaluation_extra_conflict(claim, facts, public.get('log', [])):
                contradicted.append(claim.get('text') or str(claim.get('value')))
                continue
            # Supporting context may be correct without appearing in the reference query.
            # Profit/cost/margin cannot be inferred from this revenue-only dataset.
            if re.search(r'\b(profit|margin|cost)\b', str(claim.get('text') or ''), re.I) and not re.search(r'profit|margin|cost', str(claim.get('column') or ''), re.I):
                contradicted.append(claim.get('text'))
            continue
        calc = claim.get('calc', 'none')
        inputs = claim.get('inputs')
        evidence = numbers_in_log(public.get('log', []))
        if calc != 'none' and inputs and all(which_call(v, evidence) is not None for v in inputs) and evaluation_same(claim.get('value'), recalculate(calc, inputs)):
            continue
        unassessed.append(claim.get('text') or str(claim.get('value')))
    public['claims'] = bound
    undeclared = undeclared_numbers(public)
    prose_conflicts = evaluation_prose_conflicts(public, facts)
    contradicted.extend(prose_conflicts)
    failed = bool(answer.get('error') or answer.get('insufficient_data'))
    correct = bool(required_ok and not failed and not contradicted and not unassessed and not undeclared)
    checked = verify(deepcopy(answer))
    stats = score(checked)
    displayed_checked = verify(deepcopy(public))
    displayed = displayed_checked.get('claims', [])
    display_bad = sum(c.get('status') != 'supported' for c in displayed)
    return {**stats, **evaluation_tools(answer, question),
            'correct': float(correct) if question.get('answerable') else None,
            'requested_facts_correct': float(required_ok and not failed) if question.get('answerable') else None,
            'contradicted_claims': len(contradicted), 'unassessed_extra_claims': len(unassessed),
            'undeclared_displayed_numbers': len(undeclared),
            'prose_conflicts': prose_conflicts,
            'factual_errors': contradicted, 'unassessed_claims': unassessed,
            'displayed_claims': len(displayed), 'displayed_supported': len(displayed) - display_bad,
            'displayed_unsupported': display_bad,
            'displayed_unsupported_rate': display_bad / len(displayed) if displayed else None,
            'refused': bool(answer.get('insufficient_data')), 'retries': answer.get('retries', 0),
            'route': answer.get('route', 'model' if answer.get('tier') == 1 else 'model_tools'),
            'api_cost_usd': 0.0}

def is_correct(row):
    """Compatibility for old saved scalar records; new entry points use score_answer."""
    if not row['answerable']:
        return None
    if (isinstance(row.get('error'), str) and row['error']) or row.get('refused'):
        return 0.0
    if isinstance(row.get('answer'), dict):
        return score_answer(row['answer'], row.get('question_spec', row), row.get('expected_rows'))['correct']
    if isinstance(row.get('expected_rows'), list):
        return float(result_matches(row.get('visible_claims', []), row['expected_rows'], row.get('log', [])))
    targets = [row['gt_value']] + (row.get('also_accept') or [])
    return float(any(about_equal(v, target) for v in row['values'] for target in targets))

def evaluation_expectations(question):
    """Read reference results only in evaluation, never in inference."""
    if not question.get('gt_sql') or not question.get('answerable'):
        return []
    with duckdb.connect(DB, read_only=True, config={'enable_external_access': 'false'}) as con:
        cursor = con.execute(question['gt_sql'])
        columns = [column[0] for column in cursor.description]
        return [{k: json_safe(v) for k, v in zip(columns, row)} for row in cursor.fetchall()]

def evaluation_tools(answer, question):
    """Separate tool choice from execution and only score applicable charts."""
    calls = answer.get('log', [])
    tools = {c['tool'] for c in calls}
    expected_tool = question.get('expected_tool', 'none')
    refusal_rule = not question.get('answerable') and answer.get('route') == 'rules_refusal'
    tool_ok = (not tools if expected_tool == 'none' else expected_tool in tools)
    chart = answer.get('chart')
    expected_chart = question.get('expected_chart', 'none')
    chart_applicable = expected_chart != 'none'
    chart_ok = None
    if chart_applicable:
        chart_ok = False
        if chart:
            checked = make_chart(chart, list(calls))
            chart_ok = bool(checked['ok'] and chart.get('type') == expected_chart)
    return {'tool_selection_correct': None if refusal_rule else float(tool_ok),
            'chart_applicable': chart_applicable,
            'chart_correct': float(chart_ok) if chart_applicable else None,
            'refusal_correct': float(refusal_is_correct(answer)) if not question.get('answerable') else None,
            **{key: answer.get('usage', {}).get(key) for key in ('model_calls', 'input_tokens', 'output_tokens')}}

def evaluation_summary(frame):
    """Report explicit numerators/denominators, including routes and SQL attempts."""
    import pandas as pd
    output = []
    for tier, group in frame.groupby('tier'):
        for route, selected in [('all', group)] + list(group.groupby('route')):
            answerable = selected[selected.answerable == True]
            refusals = selected[selected.answerable == False]
            sql_calls = selected['queries'].sum()
            claims = selected['displayed_claims'].sum()
            output.append({'tier': tier, 'route': route, 'cases': len(selected),
                'answerable_cases': len(answerable), 'correct_answers': answerable.correct.sum(),
                'accuracy': answerable.correct.mean(),
                'refusal_cases': len(refusals), 'refused_tricks': refusals.refusal_correct.sum(),
                'sql_attempts': sql_calls, 'sql_successes': selected['sql_successes'].sum(),
                'queries_ok': selected['sql_successes'].sum() / sql_calls if sql_calls else None,
                'displayed_claims': claims, 'displayed_supported': selected['displayed_supported'].sum(),
                'evidence_coverage': selected['displayed_supported'].sum() / claims if claims else None,
                'unsupported_rate': selected['displayed_unsupported'].sum() / claims if claims else None,
                'required_charts': selected.chart_correct.notna().sum(),
                'chart_correct': selected.chart_correct.mean(),
                'tool_selection': selected.tool_selection_correct.mean(),
                'avg_retries': selected.retries.mean(), 'avg_seconds': selected.seconds.mean(),
                'api_cost_usd': selected.api_cost_usd.sum()})
    return pd.DataFrame(output)

def export_evaluation(frame, destination):
    """One exporter is shared by notebook, live runner and historical rescoring."""
    from pathlib import Path
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    directory = Path(destination)
    directory.mkdir(parents=True, exist_ok=True)
    summary = evaluation_summary(frame)
    for name, table in [('all_runs.csv', frame), ('summary.csv', summary)]:
        temporary = directory / (name + '.tmp')
        table.to_csv(temporary, index=False)
        temporary.replace(directory / name)
    overview = summary[summary.route == 'all'].set_index('tier')
    fig = Figure(figsize=(13, 3.5))
    FigureCanvasAgg(fig)
    axes = fig.subplots(1, 3)
    for axis, metric, title in zip(axes, ['accuracy', 'unsupported_rate', 'avg_seconds'],
            ['Requested facts and extra-claim checks', 'Unsupported displayed claims', 'Seconds per answer']):
        overview[metric].astype(float).plot.bar(ax=axis, color='#315440')
        axis.set_title(title)
        axis.set_xlabel('Tier')
        if metric != 'avg_seconds':
            axis.set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(directory / 'comparison.png', dpi=150)
    return summary

def refusal_is_correct(answer):
    """A refusal flag cannot excuse invented figures; allow verified context."""
    from copy import deepcopy
    if not answer.get('insufficient_data'):
        return False
    public = deepcopy(answer)
    if answer.get('tier') == 3:
        public['claims'] = claims_to_show(public)
    checked = verify(public)
    return (not checked.get('undeclared') and not checked.get('reconciliation')
            and all(c.get('status') == 'supported' for c in checked.get('claims', [])))

results_dir = Path('../eval/results/notebook_run')
results_dir.mkdir(parents=True, exist_ok=True)
rows = []
for i, question in enumerate(questions, 1):
    for tier, function in [(1, tier1), (2, tier2), (3, tier3)]:
        print('[%2d/%d] %s tier %d' % (i, len(questions), question['id'], tier), end=' ', flush=True)
        started = time.time()
        row = {'id': question['id'], 'tier': tier, 'difficulty': question['difficulty'],
               'answerable': question['answerable'], 'reviewed_by': question.get('reviewed_by'),
               'error': None, 'usefulness_rating': None}
        try:
            expected = evaluation_expectations(question)
            answer = function(question['question'])
            answer['tier'] = tier
            row.update(score_answer(answer, question, expected))
            visible = claims_to_show(answer) if tier == 3 else answer.get('claims', [])
            row.update(values=[c.get('value') for c in visible], log=answer.get('log', []))
            print('ok')
        except Exception as error:
            answer = {'tier': tier, 'error': str(error), 'claims': [], 'log': []}
            row.update(score_answer(answer, question, []))
            row['error'] = str(error)
            print('ERROR:', error)
        row['seconds'] = round(time.time() - started, 1)
        rows.append(row)
        pd.DataFrame(rows).to_csv(results_dir / 'all_runs.csv', index=False)
df = pd.DataFrame(rows)
summary = export_evaluation(df, results_dir)
print('saved', len(df), 'rows; errors:', df.error.notna().sum())
print('Exports:', results_dir)


def evaluation_metric(value):
    """Recognise common measurement kinds, without guessing unknown SQL aliases."""
    text = str(value or '').lower().replace('_', ' ').strip()
    if re.search(r'%|\bpct\b|percent|percentage|\bshare\b|\brate\b|\bratio\b', text):
        return 'percentage'
    if re.search(r'average order value|\baov\b|revenue|\bgbp\b|£|\bpounds?\b', text):
        if re.search(r'per\s+(?:trading\s+)?day|/\s*day|\bdaily\b', text):
            return 'money_per_day'
        return 'money'
    if re.search(r'trading days?|\bdays?\b', text):
        return 'days'
    if re.search(r'\borders?\b|\binvoices?\b', text):
        return 'orders'
    if re.search(r'\bunits?\b|\bquantity\b', text):
        return 'units'
    if re.search(r'\bcustomers?\b|\bbuyers?\b', text):
        return 'customers'
    return None

def evaluation_metric_problem(claim, fact):
    """A matching number must not change from revenue to an order count."""
    expected = evaluation_metric(fact['column'])
    if expected is None:
        return False
    text = str(claim.get('text') or '')
    unit = evaluation_metric(claim.get('unit'))
    # GBP is a valid currency label for a per-day amount; the column/text
    # supplies its rate dimension. An explicit per-day unit is not a total.
    if unit is not None and unit != expected and not (expected == 'money_per_day' and unit == 'money'):
        return True
    # The currency symbol and percentage marker attach directly to a value.
    # Otherwise ignore identifier prefixes ('Customer 18102: 145 orders').
    visible = re.sub(r'\bcustomer(?:_id)?\s+\d+\s*[:;,.]?', '', text, flags=re.I)
    kind = evaluation_metric(visible)
    return kind is not None and kind != expected and not (expected == 'money_per_day' and kind == 'money')

def evaluation_scope_labels(row, call=None):
    """Labels distinguish optional breakdowns from conflicting aggregate totals."""
    labels = {key: value for key, value in row.items()
              if key in ('country', 'invoice_month', 'customer_id', 'stock_code', 'description')}
    if call:
        try:
            tree = sqlglot.parse_one(call.get('code') or '', read='duckdb')
            for where in tree.find_all(exp.Where):
                for node in where.find_all(exp.EQ):
                    if isinstance(node.this, exp.Column) and isinstance(node.expression, exp.Literal):
                        if node.this.name in ('country', 'invoice_month', 'customer_id', 'stock_code'):
                            labels.setdefault(node.this.name, node.expression.this)
        except Exception:
            pass
    return labels

def evaluation_extra_conflict(claim, facts, log):
    """Do not let a SQL pointer validate a second contradictory headline value.

    A labelled month/country/customer breakdown can be legitimate optional
    context. An unqualified second total of the same metric cannot be excused
    merely because a query returned that number.
    """
    found, _ = claim_cell(claim, log)
    if not found:
        return False
    source = next(call for call in log if call.get('n') == claim['from_call'] and call.get('ok') and call.get('tool') == 'run_sql')
    source_row = source['rows'][claim['row']]
    column = claim['column']
    own_fact = {'column': column}
    if evaluation_metric_problem(claim, own_fact):
        return True
    metric = evaluation_metric(column)
    comparable = [fact for fact in facts if metric is not None and evaluation_metric(fact['column']) == metric]
    if not comparable:
        return False
    labels = evaluation_scope_labels(source_row, source)
    # Other explicit scopes must be visible; they cannot be hidden in the SQL.
    visible_labels = {key: value for key, value in labels.items()
                      if evaluation_has_label(claim.get('text', ''), evaluation_labels(key, value))}
    for fact in comparable:
        expected_labels = evaluation_scope_labels(fact['row'])
        if visible_labels and visible_labels != expected_labels:
            continue
        if not evaluation_same(claim.get('value'), fact['value']):
            return True
    return False

def evaluation_prose_conflicts(answer, facts, text=None):
    """Check numeric prose against displayed fact labels and measurement kinds.

    This is a bounded consistency check, not unrestricted language understanding.
    Local clauses catch known label swaps; a bare follow-up sentence can inherit
    an entity introduced elsewhere in the findings.
    """
    if text is None:
        # Keep fields separate: a correct label in the findings must not excuse
        # a contradictory label in a newly displayed explanation.
        problems = []
        for field in public_prose_texts(answer):
            problems.extend(evaluation_prose_conflicts(answer, facts, field))
        return list(dict.fromkeys(problems))
    if not text:
        return []
    problems = []
    clauses = re.split(r'(?<=[.;!?])\s+|,(?=\s+[A-Za-z£])\s*|\n+|\s+(?:and|while|whereas|against|versus)\s+', text, flags=re.I)
    for clause in clauses:
        for start, end, value in number_tokens(clause):
            if is_date_context(clause, start, end) or is_evidence_label(clause, start, end, answer):
                continue
            candidates = [fact for fact in facts if evaluation_same(value, fact['value'])]
            if not candidates:
                continue
            metric_candidates = [fact for fact in candidates if not evaluation_metric_problem({'text': clause}, fact)]
            if not metric_candidates:
                problems.append(clause.strip())
                continue
            matched = False
            for fact in metric_candidates:
                valid = True
                for kind, choices in fact['labels']:
                    competitors = [labels for other in facts for other_kind, labels in other['labels']
                                   if other_kind == kind and not set(labels).intersection(choices)]
                    if any(evaluation_has_label(clause, labels) for labels in competitors):
                        valid = False
                    elif not evaluation_has_label(clause, choices) and not evaluation_has_label(text, choices):
                        # A yes/no question about a named month may answer 'It has
                        # eight trading days' without repeating the requested date.
                        named_period = kind == 'invoice_month' and evaluation_has_label(answer.get('question', ''), choices)
                        shared_days = (kind == 'invoice_month' and evaluation_metric(fact['column']) == 'days'
                                       and re.search(r'\b(?:both|each|same|equal)\b', clause, re.I)
                                       and evaluation_has_label(answer.get('findings', ''), choices))
                        if not named_period and not shared_days:
                            valid = False
                if valid:
                    matched = True
                    break
            if not matched:
                problems.append(clause.strip())
    return list(dict.fromkeys(problems))


## 2. Was the answer right?

Correct means any number the tier gave is within 1% of the answer we worked
out ourselves in SQL.


In [ ]:
# Accuracy already uses the common score_answer rubric.
answerable = df[df.answerable].copy()
tricks = df[~df.answerable].copy()
df[['id', 'tier', 'correct', 'requested_facts_correct', 'error']].head(12)


## 3. The scoreboard


In [ ]:
# Every route has its own denominator; empty answers have no claim denominator.
summary = evaluation_summary(df)
summary


## 4. Accuracy by difficulty


In [ ]:
(answerable.pivot_table(index='difficulty', columns='tier',
                       values='correct', aggfunc='mean')
           .reindex(['easy', 'medium', 'hard'])
           .round(2))


## 5. The five impossible questions

The clearest table in the project. Anything that is not a refusal is a
made-up number a manager would have acted on.


In [ ]:
tricks.pivot_table(index='id', columns='tier', values='refusal_correct')


## 6. Charts for the report


In [ ]:
# The shared exporter creates the same CSV summaries and plot as the live runner.
summary = export_evaluation(df, results_dir)
from IPython.display import Image, display
display(Image(filename=str(results_dir / 'comparison.png')))


## 7. What this shows

Fill in once you have the real numbers:

- Tier 1 to Tier 2: accuracy went from ___ to ___ (what using the real data
  is worth)
- Tier 2 to Tier 3: made-up claims went from ___ to ___ (what the checking
  is worth)
- Tier 3 took ___ seconds against Tier 2's ___

**Say the cost out loud.** "Slower but much more accurate" is a believable
finding. Pretending there is no trade-off is not.


## Regression examples after the reliability fixes
These common analyses use checked query patterns in both grounded tiers. Tier 3 additionally verifies the returned facts. Free-form questions still use Gemma.


In [ ]:
# These checks query the real database; no model or benchmark answers are loaded.
example_questions = [
    "What was our total revenue in 2011?",
    "Which five products generated the most revenue in 2011?",
    "Did November 2011 beat October 2011 on revenue, and by what percentage?",
    "Is December 2011 a complete month in the dataset?",
    "What was our cancellation rate in 2011?",
]
for question in example_questions:
    result = tier3(question)
    print(question)
    print(result['findings'])
    print('What this means:', result['interpretation'])
    for step in result['suggested_next_steps']:
        print('Next:', step)
    print("Claims:", len(result['claims']), "| Chart:", (result['chart'] or {}).get('type', 'none'))
    print()


What was our total revenue in 2011?
Revenue was £9,809,614.01. Cancellations are excluded.
What this means: This is the sales baseline for the requested scope. A monthly breakdown shows whether activity is steady or concentrated in particular periods.
Next: How did revenue change by month in 2011?
Next: Which products contributed the most revenue in 2011?
Claims: 1 | Chart: none

Which five products generated the most revenue in 2011?
REGENCY CAKESTAND 3 TIER leads among the products shown, with £146,461.78 in revenue. The full ranking is available in the detailed figures. Cancellations are excluded.
What this means: This identifies the strongest recorded products for the selected measure. Use it as a starting point for stock review alongside cancellation values and product costs.
Next: Which products sold the most units in 2011?
Next: Review cancellation values before changing stock priorities.
Claims: 5 | Chart: bar

Did November 2011 beat October 2011 on revenue, and by what percent